# Setup

In [ ]:
library(phyloseq)
library(tidyverse)
library(RColorBrewer)
library(DivNet)
library(radEmu)
library(readxl)
library(stringr)
library(janitor)
library(ape)
library(scales)
library(patchwork)
library(forcats)
library(writexl)

In [ ]:
conflicted::conflicts_prefer(dplyr::filter, dplyr::select, dplyr::lag, dplyr::recode)

In [ ]:
propagate_incomplete_taxonomy <- function(phyloseq){
  taxranks <- colnames(tax_table(phyloseq))
  tt <- matrix(as.character(tax_table(phyloseq)@.Data),          
               nrow = ntaxa(phyloseq),
               dimnames = dimnames(tax_table(phyloseq)))
  for (i in 2:length(taxranks)) {
    na_idx <- is.na(tt[, taxranks[i]])
    tt[na_idx, taxranks[i]] <- paste0(tt[na_idx, taxranks[i - 1]], "_spc")
  }
  tax_table(phyloseq) <- tax_table(tt)
  return(phyloseq)
}

In [ ]:
# display-only relabelling: expand propagated placeholders, tidy SILVA underscores
prettify_taxa <- function(physeq) {
  tt <- as.matrix(tax_table(physeq))
  tt <- gsub("_spc(_spc)*$", " (unassigned)", tt)
  tt <- gsub("_", " ", tt)
  tax_table(physeq) <- tax_table(tt)
  physeq
}

In [ ]:
norm_key <- function(x) {
  out <- str_replace_all(str_to_lower(as.character(x)), "[^a-z]", "")
  out[is.na(x)] <- NA_character_
  out
}
sid_norm <- function(x) str_remove(str_trim(as.character(x)), "-P$")

In [ ]:
meta_dir   <- "../metadata"
out_dir    <- "../analysis/output"

## Add metadata

In [ ]:

f_meta  <- file.path(meta_dir, "Case and control data 10 July 2026.xlsx")


In [ ]:
raw <- read_excel(f_meta, sheet = "Cases", .name_repair = "unique")

sid <- grep("STUDY ID", names(raw), value = TRUE)
stopifnot("Cases: expected >=2 STUDY ID columns from side-by-side blocks" =
            length(sid) >= 2)
key <- lapply(sid, function(c) str_trim(as.character(raw[[c]])))
if (!all(vapply(key[-1], identical, logical(1), key[[1]])) || anyNA(key[[1]]))
  stop("Cases: STUDY ID columns are not row-identical - blocks misaligned, aborting read.")

# report which columns were dropped as completely empty
dropped <- names(raw)[vapply(raw, function(x) all(is.na(x)), logical(1))]
if (length(dropped)) message("Cases: all-empty columns dropped: ", toString(dropped))  


sepsis <- raw %>%
  rename(study_id = all_of(sid[1])) %>%
  select(-all_of(sid[-1])) %>%
  select(where(~ !all(is.na(.x)))) %>%
  clean_names() %>%
  mutate(study_id = str_trim(study_id), group = "sepsis")

In [ ]:
head(sepsis)

In [ ]:
controls <- read_excel(f_meta, sheet = "Health control", .name_repair = "unique") %>%
  clean_names() %>%
  mutate(study_id = str_trim(study_id), group = "control")

In [ ]:
head(controls)

In [ ]:
names(sepsis)

In [ ]:
names(controls)

### adjust variable names

In [ ]:
# metadata harmonization: sepsis (Cases) + controls (Health) 
# ---------------------------------------------------------------------------
# 1. variable crosswalk: target <- (sepsis, controls). NA = absent in that cohort.
# ---------------------------------------------------------------------------
xwalk <- tibble::tribble(
  ~target,                              ~sepsis,                           ~controls,
  # --- identifiers / admission ---
  "study_id",                           "study_id",                        "study_id",
  "reg_no",                             "reg_no",                          NA,
  "mother_id",                          NA,                                "mother_id",
  "admission_date",                     "admission_date",                  NA,
  "ward",                               "ward",                            NA,
  # --- demographics / birth ---
  "date_of_birth",                      "date_of_birth",                   "dob",
  "age_in_days",                        "age_in_days",                     "age_in_days",
  "sex",                                "sex",                             "sex",
  "born_in_health_facility",            "born_in_health_facility",         "born_in_hf",
  "facility_name",                      "facility_name",                   "facility_name",
  "mode_of_delivery",                   "mode_of_delivery",                "mod",
  "gestation_age_wk",                   "gestation_age",                   "ga_by_finns",
  "birth_weight_g",                     "birth_weight_in_gms",             "bwt_in_gms",
  "apgar_1min",                         "apgar_1st",                       "x1st_apgar",
  "apgar_5min",                         "apgar_5th",                       "x5th_apgar",
  "cry_immediately",                    "cry_immediately",                 "cry_immediately",
  "meconium_stain",                     "meconium_stain",                  "meconium_stain",
  "rupter_of_membrane_duration",        "rupter_of_membrane_duration",     "rom_duration",
  # --- maternal / perinatal antibiotics (structural mismatch: dated in controls only) ---
  "antibiotics_during_pregnancy",       "antibiotics_during_pregnancy",    "antibiotics_during_pregnancy",
  "antibiotics_perinatal",              "perinatal_antibiotics",           NA,
  "antibiotics_mother_route",          "mode_of_administration",          "mode_of_administration",
  "antibiotics_mother_name",            "antibiotic_name",                 "antibiotic_name",
  "antibiotics_mother_start",           NA,                                "start_date",
  "antibiotics_mother_end",             NA,                                "end_date",
  "perinatal_fever",                    "perinatal_fever",                 "perinatal_fever",
  "maternal_fever",                     "maternal_hx_of_fever",            "maternal_hx_of_fever",
  # --- admission vitals / anthropometry ---
  "temp_c",                             "temp",                            "temp",
  "heart_rate_bpm",                     "heart_rate_beatspmin",            "hr",
  "resp_rate_permin",                  "respiratory_rate_breathpermin",   "rr",
  "spo2",                               "spo2",                            "spo2",
  "random_blood_glucose",               "random_blood_glucose",            "rbg",
  "length_cm",                          "length_in_cm",                    "length_in_cm",
  "occipital_frontal_circumference_cm", "occipital_frontal_circumferance", "ofc",
  "admission_weight_g",                 "weight_in_gms",                   NA,
  "oxygen_mask",                        "oxygen_mask",                     "oxygen_mask",
  "feeding",                            "feeding",                         "feeding",
  # --- outcome ---
  "outcome",                            "outcome_111",                     NA,
  "outcome_date",                       "d_o_outcome",                     NA,
  "hospital_stay",                      "hospital_stay_113",               NA,
  "discharge_date",                     NA,                                "date_of_discharge",
  "followup_date",                      NA,                                "date_of_follow_up",

  # --- SEPSIS-ONLY: admission symptoms (Y/N; not temporal -> own names) ---
  "difficulty_breathing",       "difficult_in_breathing",          NA,
  "fever",                      "fever",                           NA,
  "convulsion",                 "convulsion",                      NA,
  "inability_to_breastfeed",    "inability_to_breastfeed",         NA,
  "jaundice",                   "jaundice",                        NA,
  "cyanosis",                   "cyanosis",                        NA,
  "preterm",                    "preterm",                         NA,
  "vomiting",                   "vomiting",                        NA,
  "diarrhea",                   "diarrhea",                        NA,
  "irritability",               "irritability",                    NA,
  "symptom_other",                      "others_23",                       NA,
  "symptom_other_text",                 "mention_24",                      NA,
  # --- SEPSIS-ONLY: devices ---
  "catheter",                    "catheter",                        NA,
  "iv_line",                     "iv_line",                         NA,
  "nasogastric_tube",            "nasogastric_tube_in_sit",         NA,
  # --- SEPSIS-ONLY: admission diagnoses (Y/N; not temporal -> own names) ---
  "early_onset_sepsis",              "early_onset_sepsis",              NA,
  "respiratory_distress_syndrome",   "respiratory_distress_syndrome",   NA,
  "transient_tachypnea_newborn",     "transient_tachypnea_of_the_newborn", NA,
  "neonatal_jaundice",               "neonatal_jaundice",               NA,
  "mild_birth_asphyxia",             "mild_birth_asphyxia",             NA,
  "severe_birth_asphyxia",           "severe_birth_asphyxia",           NA,
  "premature_lbw",                   "premature_with_lbw",              NA,
  "premature_vlbw",                  "premature_with_vlbw",             NA,
  "congenital_heart_disease",        "congenital_heart_disease",        NA,
  "congenital_pneumonia",            "congenital_pneumonia",            NA,
  "meconium_aspiration",             "meconium_aspiration",             NA,
  "dx_other",                           "others_61",                       NA,
  "dx_other_text",                      "mention_62",                      NA,
  "at_discharge",                    "dx_at_discharge",                 NA,
  # --- SEPSIS-ONLY: ward antibiotic courses (temporal -> numbered) ---
  "antibiotics_ward_given",             "antibiotics_in_ward",             NA,
  "antibiotics_ward_1_name",            "x1st_name_65",                    NA,
  "antibiotics_ward_1_dose",            "dose_66",                         NA,
  "antibiotics_ward_1_start",           "start_67",                        NA,
  "antibiotics_ward_1_end",             "end_68",                          NA,
  "antibiotics_ward_2_name",            "x2nd_name_69",                    NA,
  "antibiotics_ward_2_dose",            "dose_70",                         NA,
  "antibiotics_ward_2_start",           "start_71",                        NA,
  "antibiotics_ward_2_end",             "end_72",                          NA,
  "antibiotics_ward_3_name",            "x3rd_name_73",                    NA,
  "antibiotics_ward_3_dose",            "dose_74",                         NA,
  "antibiotics_ward_3_start",           "start_75",                        NA,
  "antibiotics_ward_3_end",             "end_76",                          NA,
  "antibiotics_change_given",           "antibiotic_change",               NA,
  "antibiotics_change_1_name",          "x1st_name_78",                    NA,
  "antibiotics_change_1_dose",          "dose_79",                         NA,
  "antibiotics_change_1_start",         "start_80",                        NA,
  "antibiotics_change_1_end",           "end_81",                          NA,
  "antibiotics_change_2_name",          "x2nd_name_82",                    NA,
  "antibiotics_change_2_dose",          "dose_83",                         NA,
  "antibiotics_change_2_start",         "start_84",                        NA,
  "antibiotics_change_2_end",           "end_85",                          NA,
  # --- SEPSIS-ONLY: serial CRP (temporal -> numbered) ---
  "c_reactive_protein_1",               "x1st_crp",                        NA,
  "c_reactive_protein_2",               "x2nd_crp",                        NA,
  "c_reactive_protein_3",               "x3rd_crp",                        NA,
  # --- SEPSIS-ONLY: serial CBC (temporal -> analyte + timepoint) ---
  "leukocytes_1",                       "leukocyte_1",                     NA,
  "neutrophils_1",                      "neutrophils_1",                   NA,
  "lymphocytes_1",                      "lymph_1",                         NA,
  "monocytes_1",                        "mono_1",                          NA,
  "hemoglobin_1",                       "hb_1",                            NA,
  "platelets_1",                        "plt_1",                           NA,
  "leukocytes_2",                       "leuc_2",                          NA,
  "neutrophils_2",                      "neutr_2",                         NA,
  "lymphocytes_2",                      "lymph_2",                         NA,
  "monocytes_2",                        "mono_2",                          NA,
  "hemoglobin_2",                       "hb_2",                            NA,
  "platelets_2",                        "plt_2",                           NA,
  "leukocytes_3",                       "leuc_3",                          NA,
  "neutrophils_3",                      "neutr_3",                         NA,
  "lymphocytes_3",                      "lymph_3",                         NA,
  "monocytes_3",                        "mono_3",                          NA,
  "hemoglobin_3",                       "hb_3",                            NA,
  "platelets_3",                        "plt_3",                           NA,
  # --- SEPSIS-ONLY: blood culture + antibiogram (per drug: zone mm + S/I/R) ---
  "blood_culture_result",               "culture_results",                 NA,
  "blood_culture_isolate",              "isolate",                         NA,
  "blood_culture_gram_stain",           "gram",                            NA,
  "ampicillin_zone_mm",     "ampicillin_zone",                 NA,
  "ampicillin_sir",         "ampicillin",                      NA,
  "cefoxitin_zone_mm",      "cefoxitin_zone",                  NA,
  "cefoxitin_sir",          "cefoxitin",                       NA,
  "gentamicin_zone_mm",     "gentamicin_zone",                 NA,
  "gentamicin_sir",         "genta",                           NA,
  "ciprofloxacin_zone_mm",  "cipro_zone",                      NA,
  "ciprofloxacin_sir",      "cipro",                           NA,
  "piperacillin_tazobactam_zone_mm", "pipera_tazobactam_zone", NA,
  "piperacillin_tazobactam_sir",     "piper_tazobac",          NA,
  "amoxicillin_clavulanate_zone_mm", "amoxclav_zone",          NA,
  "amoxicillin_clavulanate_sir",     "amoxiclav",              NA,
  "ceftriaxone_zone_mm",    "ceftriaxone_zone",                NA,
  "ceftriaxone_sir",        "ceftriaxone",                     NA,
  "ceftazidime_zone_mm",    "ceftazidime_zone",                NA,
  "ceftazidime_sir",        "ceftazidime",                     NA,
  "amikacin_zone_mm",       "amikacin_zone",                   NA,
  "amikacin_sir",           "amikacin",                        NA,
  "meropenem_zone_mm",      "meropenem_zone",                  NA,
  "meropenem_sir",          "meropenem",                       NA,
  "chloramphenicol_zone_mm","chrolamphenical_zone",            NA,
  "chloramphenicol_sir",    "chloramphenical",                 NA,
  "clindamycin_zone_mm",    "clindamycin_zone",                NA,
  "clindamycin_sir",        "clindamycin",                     NA,
  "erythromycin_zone_mm",   "erythromycin_zone",               NA,
  "erythromycin_sir",       "erythro",                         NA,
  "trimethoprim_sulfamethoxazole_zone_mm", "trimethoprim_zone", NA,
  "trimethoprim_sulfamethoxazole_sir",     "trime_sulpha",      NA,
  "tetracycline_zone_mm",   "tetracyclin_zone",                NA,
  "tetracycline_sir",       "tetracycline",                    NA,
  "vancomycin_zone_mm",     "vancomycin_zone",                 NA,
  "vancomycin_sir",         "vancomycin",                      NA,
  "colistin_zone_mm",       "colistin_zone",                   NA,
  "colistin_sir",           "colistin",                        NA,
  "esbl",                   "esbl",                            NA,
  "stool_c1",   "stool_c1",   NA,
  "stool_c2",   "stool_c2",   NA,
  "stool_c3",   "stool_c3",   NA,
  "stool_c4",   "stool_c4",   NA,
  # --- SEPSIS-ONLY: daily follow-up day 1-7 (temporal -> numbered) ---
  "age_1",                     "age_day_1",                       NA,
  "new_complaints_1",          "new_complaints_166",              NA,
  "weight_g_1",                "weight_1",                        NA,
  "temp_c_1",                  "temp_1",                          NA,
  "random_blood_glucose_1",    "rbg_1",                           NA,
  "spo2_1",                    "spo2_1",                          NA,
  "resp_rate_1",               "rr_1",                            NA,
  "age_2",                     "age_day_2",                       NA,
  "new_complaints_2",          "new_complaints_173",              NA,
  "weight_g_2",                "weight_2",                        NA,
  "temp_c_2",                  "temp_2",                          NA,
  "random_blood_glucose_2",    "rbg_2",                           NA,
  "spo2_2",                    "spo2_2",                          NA,
  "resp_rate_2",               "rr_2",                            NA,
  "age_3",                     "age_day_3",                       NA,
  "new_complaints_3",          "new_complaints_180",              NA,
  "weight_g_3",                "weight_3",                        NA,
  "temp_c_3",                  "temp_3",                          NA,
  "random_blood_glucose_3",    "rbg_3",                           NA,
  "spo2_3",                    "spo2_3",                          NA,
  "resp_rate_3",               "rr_3",                            NA,
  "age_4",                     "age_day_4",                       NA,
  "new_complaints_4",          "new_complaints_187",              NA,
  "weight_g_4",                "weight_4",                        NA,
  "temp_c_4",                  "temp_4",                          NA,
  "random_blood_glucose_4",    "rbg_4",                           NA,
  "spo2_4",                    "spo2_4",                          NA,
  "resp_rate_4",               "rr_4",                            NA,
  "age_5",                     "age_day_5",                       NA,
  "new_complaints_5",          "new_complaints_194",              NA,
  "weight_g_5",                "weight_5",                        NA,
  "temp_c_5",                  "temp_5",                          NA,
  "random_blood_glucose_5",    "rbg_5",                           NA,
  "spo2_5",                    "spo2_5",                          NA,
  "resp_rate_5",               "rr_5",                            NA,
  "age_6",                     "age_day_6",                       NA,
  "new_complaints_6",          "new_complaints_201",              NA,
  "weight_g_6",                "weight_6",                        NA,
  "temp_c_6",                  "temp_6",                          NA,
  "random_blood_glucose_6",    "rbg_6",                           NA,
  "spo2_6",                    "spo2_6",                          NA,
  "resp_rate_6",               "rr_6",                            NA,
  "age_7",                     "age_day_7",                       NA,
  "new_complaints_7",          "new_complaints_208",              NA,
  "weight_g_7",                "weight_7",                        NA,
  "temp_c_7",                  "temp_7",                          NA,
  "random_blood_glucose_7",    "rbg_7",                           NA,
  "spo2_7",                    "spo2_7",                          NA,
  "resp_rate_7",               "rr_7",                            NA
)

In [ ]:
sepsis$outcome_111

In [ ]:
sepsis$outcome_214

In [ ]:
sepsis$hospital_stay_113

In [ ]:
sepsis$hospital_stay_215

In [ ]:
# ---------------------------------------------------------------------------
# 2. drop redundant columns
# ---------------------------------------------------------------------------
sepsis <- sepsis %>%
  select(-any_of(c("outcome_214", "hospital_stay_215")))   # keep _111 / _113, drop the rest

In [ ]:
# ---------------------------------------------------------------------------
# 3. rename each cohort to the shared target schema
# ---------------------------------------------------------------------------
rename_from_xwalk <- function(df, xwalk, from) {
  m    <- xwalk %>% filter(!is.na(.data[[from]]))
  ren  <- setNames(m[[from]], m$target)
  miss <- setdiff(unname(ren), names(df))
  if (length(miss)) stop(sprintf("[%s] source columns not found: %s", from, toString(miss)))
  extra <- setdiff(names(df), unname(ren))
  if (length(extra)) message(sprintf("[%s] columns not in crosswalk, dropped: %s", from, toString(extra)))
  df %>% rename(!!!ren) %>% select(all_of(m$target))
}

# guard: every referenced source column must exist in each cohort
stopifnot(
  "sepsis cols missing"   = length(setdiff(na.omit(xwalk$sepsis),   names(sepsis)))   == 0,
  "controls cols missing" = length(setdiff(na.omit(xwalk$controls), names(controls))) == 0
)

sepsis_std   <- sepsis   %>% rename_from_xwalk(xwalk, "sepsis")   %>% mutate(group = "sepsis")
controls_std <- controls %>% rename_from_xwalk(xwalk, "controls") %>% mutate(group = "control")

### clean data types

apgar

In [ ]:
# turn non-numeric APGAR placeholders into explicit NA, then coerce cleanly
sepsis_std <- sepsis_std %>%
  mutate(across(c(apgar_1min, apgar_5min),
                ~ suppressWarnings(as.numeric(na_if(str_squish(str_to_upper(.x)), "UNKNOWN")))))

In [ ]:
sepsis_std$apgar_1min
sepsis_std$apgar_5min

date of birth

In [ ]:
# date of birth text cells are dd/m/yyyy -> force day-first, drop mdy to avoid day/month swap
to_date_mixed <- function(x, orders = c("ymd", "dmy")) {  
  if (inherits(x, "POSIXt")) return(as.Date(x))
  if (inherits(x, "Date"))   return(x)
  x <- as.character(x)
  out <- as.Date(rep(NA_real_, length(x)), origin = "1970-01-01")
  serial <- !is.na(x) & grepl("^[0-9]{5}$", x)                    # Excel serial numbers
  out[serial] <- as.Date(as.numeric(x[serial]), origin = "1899-12-30")
  txt <- !serial & !is.na(x)
  out[txt] <- as.Date(lubridate::parse_date_time(x[txt], orders = orders, quiet = TRUE))
  out
}

In [ ]:
sepsis_std <- sepsis_std %>% mutate(date_of_birth = to_date_mixed(date_of_birth))


In [ ]:
sepsis_std$date_of_birth

In [ ]:
# controls date of birth to Date format
controls_std <- controls_std %>%
  mutate(date_of_birth = to_date_mixed(date_of_birth))   # POSIXt branch -> as.Date()

In [ ]:
controls_std$date_of_birth

In [ ]:
# one row per column: name + class
tibble(
  column = names(sepsis_std),
  type   = map_chr(sepsis_std, ~ class(.x)[1])
) %>% print(n = Inf)

In [ ]:
# one row per column: name + class
tibble(
  column = names(controls_std),
  type   = map_chr(controls_std, ~ class(.x)[1])
) %>% print(n = Inf)

### merge controls and sepsis

In [ ]:
shared <- intersect(names(sepsis_std), names(controls_std))
clash <- shared[map_chr(sepsis_std[shared],   ~ class(.x)[1]) !=
                map_chr(controls_std[shared], ~ class(.x)[1])]
if (length(clash)) stop("remaining type mismatches: ", toString(clash))   # should be empty now

meta <- bind_rows(sepsis_std, controls_std)
cat("dim(meta):", dim(meta), "\n")

In [ ]:
colnames(meta)

### adjust data types

dates

In [ ]:
date_cols <- c("date_of_birth", "admission_date", "outcome_date",
               "discharge_date", "followup_date",
               grep("^antibiotics_(ward|change)_[0-9]+_(start|end)$",
                    names(meta), value = TRUE),
               "antibiotics_mother_start", "antibiotics_mother_end")

meta <- meta %>%
  mutate(across(any_of(date_cols), to_date_mixed))   # POSIXct/text -> Date

binary yes/no

In [ ]:
# inspect distinct values of the candidate yes/no columns before coercing
yn_cols <- c("difficulty_breathing","fever","convulsion","inability_to_breastfeed",
             "jaundice","cyanosis","preterm","vomiting","diarrhea","irritability",
             "cry_immediately","maternal_fever","perinatal_fever","meconium_stain",
             "catheter","iv_line","nasogastric_tube","oxygen_mask",
             "antibiotics_during_pregnancy","antibiotics_perinatal",
             "early_onset_sepsis","respiratory_distress_syndrome","transient_tachypnea_newborn",
             "neonatal_jaundice","mild_birth_asphyxia","severe_birth_asphyxia",
             "premature_lbw","premature_vlbw","congenital_heart_disease",
             "congenital_pneumonia","meconium_aspiration",
             "antibiotics_ward_given","antibiotics_change_given", "born_in_health_facility", "symptom_other", "dx_other")

meta %>% select(any_of(yn_cols)) %>%
  map(~ unique(str_squish(str_to_lower(as.character(.x))))) %>%
  tibble::enframe("column","values") %>% tidyr::unnest_wider(values, names_sep = "_")

In [ ]:
# helper: normalise yes/no text to an ordered factor, No = reference (0), Yes = 1
yn <- function(x) {
  x <- str_to_lower(str_trim(as.character(x)))
  factor(case_when(x %in% c("YES","yes", "y", "1", "true")  ~ "Yes",
                   x %in% c("NO", "no",  "n", "0", "false") ~ "No",
                   TRUE ~ NA_character_),
         levels = c("No", "Yes"))
}

In [ ]:
# yes/no -> ordered factor, No = reference (0)

meta <- meta %>%
  mutate(across(any_of(yn_cols), yn))



factors

In [ ]:
sir_cols  <- grep("_sir$",     names(meta), value = TRUE)
zone_cols <- grep("_zone_mm$", names(meta), value = TRUE)
dose_cols <- grep("_dose$",    names(meta), value = TRUE)

meta %>% count(sex)                                              # confirm Male/Female spelling
meta %>% select(all_of(sir_cols))  %>% unlist() %>% str_squish() %>% str_to_upper() %>% unique()
meta %>% select(all_of(zone_cols)) %>% unlist() %>% unique() %>%
  {.[is.na(suppressWarnings(as.numeric(.))) & !is.na(.)]} %>% unique()   # non-numeric tokens in zones
meta %>% select(all_of(dose_cols)) %>% unlist() %>% unique() %>% head(30)
meta %>% count(rupter_of_membrane_duration)                     # decide levels / ordering

In [ ]:
# esbl: distinct categorical (yes / AmpC producer / NA) -> plain factor, values preserved
meta <- meta %>%
  mutate(esbl = factor(str_squish(str_to_lower(as.character(esbl)))))

In [ ]:
# ordered clinically: Sensitive < Intermediate < Resistant. Unknown tokens -> NA (not silent).
as_sir <- function(x) {
  x <- str_squish(str_to_upper(as.character(x)))
  lvl <- dplyr::case_when(
    x %in% c("SENSITIVE", "SUSCEPTIBLE", "S") ~ "S",   
    x %in% c("INTERMEDIATE", "I")             ~ "I",   
    x %in% c("RESISTANT", "R")                ~ "R",   
    TRUE                                      ~ NA_character_
  )
  factor(lvl, levels = c("S", "I", "R"), ordered = TRUE)
}

In [ ]:
# inhibition-zone diameter (mm); non-numeric tokens ("NG","R","-") -> NA
as_zone_mm <- function(x) suppressWarnings(readr::parse_number(as.character(x)))

In [ ]:
meta <- meta %>%
  mutate(
    across(all_of(sir_cols),  as_sir),                          # S < I < R (ordered)
    across(all_of(zone_cols), as_zone_mm),                      # -> numeric mm
    across(all_of(dose_cols), ~ suppressWarnings(readr::parse_number(as.character(.x)))), # "500 mg" -> 500
    # nominal factors (R infers levels; reference only where it matters)
    across(c(facility_name, mode_of_delivery, ward, feeding,
             antibiotics_mother_route, blood_culture_result, blood_culture_gram_stain,
             outcome, group,
             blood_culture_isolate,
             rupter_of_membrane_duration,
             antibiotics_ward_1_name, antibiotics_ward_2_name, antibiotics_ward_3_name,
             antibiotics_change_1_name, antibiotics_change_2_name),
           ~ factor(str_squish(as.character(.x)))),
    sex = factor(str_to_title(str_squish(sex)), levels = c("Male", "Female"))
  )
# free text kept as character (no cast): symptom_other_text, dx_other_text,
# new_complaints_1..7, at_discharge, study_id, mother_id

In [ ]:
# what type is every column now, and which are still plain character
type_now <- tibble(column = names(meta), type = map_chr(meta, ~ class(.x)[1]))

type_now %>% count(type)                       # overview: how many of each type
type_now %>% filter(type == "character") %>% print(n = Inf)   

# Descriptive statistics

In [ ]:
# ---------------------------------------------------------------------
# 0. group factor (Sepsis left, Controls right) + derived variables
# ---------------------------------------------------------------------
meta <- meta %>%
  mutate(group = factor(as.character(group),
                        levels = c("sepsis", "control"),
                        labels = c("Sepsis", "Controls")))


In [ ]:

grp_cols <- c("Sepsis" = "#45458b", "Controls" = "#056b05")


In [ ]:
# display labels for the two groups, used in all figures
lab_bsi     <- "Neonates with BSI"
lab_healthy <- "Healthy neonates"
grp_labels  <- c(Sepsis = lab_bsi, Controls = lab_healthy)

In [ ]:
# =====================================================================
# (A) COMPARISON PLOTS  (Sepsis vs Controls)
# =====================================================================

theme_desc <- theme_minimal() +
  theme(plot.title  = element_text(size = 12, face = "bold"),
        axis.title.x = element_text(size = 10),
        axis.title.y = element_text(size = 10),
        axis.text.x  = element_text(size = 10),
        axis.text.y  = element_text(size = 10))

# numeric variables shared by both cohorts, with y-axis unit labels
num_shared <- tribble(
  ~var,                                   ~title,               ~unit,
  "gestation_age_wk",                     "Gestational age",    "weeks",
  "birth_weight_g",                       "Birth weight",       "g",
  "age_in_days",                          "Age at admission",   "days",
  "apgar_1min",                           "APGAR 1 min",        "score",
  "apgar_5min",                           "APGAR 5 min",        "score",
  "temp_c",                               "Temperature",        "\u00b0C",
  "heart_rate_bpm",                       "Heart rate",         "bpm",
  "resp_rate_permin",                    "Respiratory rate",   "/min",
  "spo2",                                 "SpO2",               "%",
  "random_blood_glucose",                 "Blood glucose",      "mmol/L",
  "length_cm",                            "Length",             "cm",
  "occipital_frontal_circumference_cm",   "Occipital frontal circumference",                "cm"
)



# categorical variables shared by both cohorts
cat_shared <- c("sex", "mode_of_delivery", "born_in_health_facility",
                "cry_immediately", "meconium_stain", "rupter_of_membrane_duration",
                "perinatal_fever", "maternal_fever", "antibiotics_during_pregnancy",
                "oxygen_mask", "feeding")

# numeric panel: boxplots, one per variable (loop instead of hard-coded p1..pN)
num_plots <- map(seq_len(nrow(num_shared)), function(i) {
  v <- num_shared$var[i]
  ggplot(meta, aes(x = group, y = .data[[v]], fill = group)) +
    geom_boxplot(outlier.size = 0.6) +
    scale_fill_manual(values = grp_cols) +
    scale_x_discrete(labels = grp_labels) +
    labs(title = num_shared$title[i], y = num_shared$unit[i], x = NULL) +
    theme_desc + theme(legend.position = "none") #+
    #ylim(0, NA)
})
num_panel <- wrap_plots(num_plots, ncol = 4)
ggsave(file.path(out_dir, "descriptive_numeric_sepsis_vs_controls.tiff"),
       num_panel, width = 11, height = 8, dpi = 300, compression = "lzw", bg = "white")
num_panel       


# categorical panel: 100%-stacked bars, one per variable
cat_plots <- map(cat_shared, function(v) {
  ggplot(meta, aes(x = group, fill = .data[[v]])) +
    geom_bar(position = "fill") +
    scale_y_continuous(labels = scales::percent) +
    scale_x_discrete(labels = grp_labels) +
    scale_fill_brewer(palette = "Greys", na.value = "grey90") +
    labs(title = str_to_title(str_replace_all(v, "_", " ")),
         x = NULL, y = "Percent", fill = NULL) +
    theme_desc + theme(legend.position = "right")
})
cat_panel <- wrap_plots(cat_plots, ncol = 3)
ggsave(file.path(out_dir, "descriptive_categorical_sepsis_vs_controls.tiff"),
       cat_panel, width = 12, height = 10, dpi = 300, compression = "lzw", bg = "white")
cat_panel       

In [ ]:
# state of the categorical variables: existence, class, level count, missingness
tibble(var = cat_shared) %>%
  mutate(exists   = var %in% names(meta),
         class    = map_chr(var, ~ if (.x %in% names(meta)) class(meta[[.x]])[1] else NA_character_),
         n_levels = map_int(var, ~ if (.x %in% names(meta)) dplyr::n_distinct(na.omit(meta[[.x]])) else NA_integer_),
         n_na     = map_int(var, ~ if (.x %in% names(meta)) sum(is.na(meta[[.x]])) else NA_integer_),
         n_rows   = nrow(meta)) %>%
  print(n = Inf)

In [ ]:
# Helper function: beautify variable names
beautify_varname <- function(varname) {
  varname %>%
    str_replace_all("_", " ") %>%            # replace _ with space
    str_replace_all("\\b([a-z])", toupper)    # capitalize first letters, preserve existing capitals
}

# Numerical summary (median [IQR]) stratified by group
num_summary <- function(var, group) {
  df <- data.frame(var = var, group = group)
  df_nonmiss <- df %>% filter(!is.na(var))
  grouped <- df_nonmiss %>%
    group_by(group) %>%
    summarise(med = median(var),
              q1 = unname(quantile(var, 0.25)),
              q3 = unname(quantile(var, 0.75)),
              .groups = "drop") %>%
    mutate(summary = sprintf("%.1f [%.1f–%.1f]", med, q1, q3)) %>%
    dplyr::select(group, summary) %>%
    pivot_wider(names_from = group, values_from = summary)
  total <- df_nonmiss %>%
    summarise(med = median(var),
              q1 = unname(quantile(var, 0.25)),
              q3 = unname(quantile(var, 0.75))) %>%
    mutate(Total = sprintf("%.1f [%.1f–%.1f]", med, q1, q3)) %>%
    dplyr::select(Total)
  n_total <- nrow(df)
  n_miss <- sum(is.na(df$var))
  p_miss <- 100 * n_miss / n_total
  missing <- sprintf("%d (%.1f%%)", n_miss, p_miss)
  bind_cols(grouped, total, tibble(Missing = missing))
}


# Categorical summary (percentages) stratified by group
cat_summary<- function(var, group, varname = NULL) {
  df <- data.frame(value = var, group = group)
  if (is.null(varname)) varname <- "Variable"
  varname <- beautify_varname(varname)
  df_nonmiss <- df %>% filter(!is.na(value))
  is_binary_yes_no <- length(unique(na.omit(var))) == 2 &&
    all(tolower(levels(factor(var))) %in% c("yes", "no"))
  grouped <- df_nonmiss %>%
    group_by(group, value) %>%
    summarise(n = n(), .groups = "drop") %>%
    group_by(group) %>%
    mutate(p = n / sum(n)) %>%
    ungroup() %>%
    mutate(summary = sprintf("%d (%.1f%%)", n, p * 100))
  if (is_binary_yes_no) {
    grouped <- grouped %>% filter(tolower(value) == "yes")
  }
  grouped_wide <- grouped %>%
    dplyr::select(value, group, summary) %>%
    pivot_wider(names_from = group, values_from = summary, values_fill = "-")
  total <- df_nonmiss %>%
    group_by(value) %>%
    summarise(n = n(), .groups = "drop") %>%
    mutate(p = n / sum(n),
           Total = sprintf("%d (%.1f%%)", n, p * 100))
  if (is_binary_yes_no) {
    total <- total %>% filter(tolower(value) == "yes")
    total$Variable <- varname
  } else {
    total$Variable <- paste0(varname, ": ", total$value)
  }
  total <- total %>% dplyr::select(value, Variable, Total)
  n_total <- nrow(data.frame(value = var))
  n_miss <- sum(is.na(var))
  p_miss <- 100 * n_miss / n_total
  total$Missing <- sprintf("%d (%.1f%%)", n_miss, p_miss)
  result <- left_join(total, grouped_wide, by = "value") %>%
    dplyr::select(Variable, everything(), -value)
  return(result)
}



# Create descriptive table in exact specified order
create_descriptive_table <- function(df, group_var, var_spec) {   
  group_var <- enquo(group_var)                                   
  group_vec <- df %>% pull(!!group_var)                           

  map_dfr(seq_len(nrow(var_spec)), function(i) {
    v    <- var_spec$var[i]
    type <- var_spec$type[i]
    if (type == "num") {
      res <- num_summary(df[[v]], group_vec)                      
      tibble(Variable = beautify_varname(v)) %>% bind_cols(res)
    } else {
      cat_summary(df[[v]], group_vec, varname = beautify_varname(v))  
    }
  }) %>%
    mutate(across(everything(), ~ replace_na(as.character(.), "-")))
}

In [ ]:
# =====================================================================
# (B) TABLE 1 
# =====================================================================

# variable order/type for Table 1 (shared variables only)
var_spec <- tribble(
  ~var,                                  ~type,
  # Demographics & birth
  "sex",                                 "cat",
  "gestation_age_wk",                    "num",
  "birth_weight_g",                      "num",
  "mode_of_delivery",                    "cat",
  "born_in_health_facility",             "cat",
  "age_in_days",                         "num",
  # Perinatal risk factors
  "apgar_1min",                          "num",
  "apgar_5min",                          "num",
  "cry_immediately",                     "cat",
  "meconium_stain",                      "cat",
  "rupter_of_membrane_duration",         "cat",
  "perinatal_fever",                     "cat",
  "maternal_fever",                      "cat",
  "antibiotics_during_pregnancy",        "cat",
  # Admission vitals & anthropometry
  "temp_c",                              "num",
  "heart_rate_bpm",                      "num",
  "resp_rate_permin",                   "num",
  "spo2",                                "num",
  "random_blood_glucose",                "num",
  "length_cm",                           "num",
  "occipital_frontal_circumference_cm",  "num",
  "oxygen_mask",                         "cat",
  "feeding",                             "cat"
)

table1 <- create_descriptive_table(df = meta, group_var = group, var_spec = var_spec)
table1
write.csv(table1[,-3], file.path(out_dir, "descriptive_sepsis_vs_controls.csv"), row.names = FALSE)



In [ ]:
# =====================================================================
# (C) SEPSIS-ONLY CHARACTERISATION (Total column only)
# =====================================================================

sepsis_only <- meta %>% filter(group == "Sepsis")

# combined ward regimen from up to three antibiotics, independent of entry order
abx_cols <- c("antibiotics_ward_1_name", "antibiotics_ward_2_name", "antibiotics_ward_3_name")
sepsis_only <- sepsis_only %>%
  mutate(across(all_of(abx_cols), ~ na_if(str_to_lower(str_squish(as.character(.x))), ""))) %>%
  rowwise() %>%
  mutate(empirical_regimen = {
    abx <- sort(unique(na.omit(c_across(all_of(abx_cols)))))
    if (length(abx) == 0) NA_character_ else paste(abx, collapse = " + ")
  }) %>%
  ungroup()

# check spelling variants before building the table
sepsis_only %>% count(empirical_regimen, name = "n") %>% arrange(desc(n))

# Numerical summary, total only (median [IQR] + Missing)
num_summary_total <- function(var) {
  df_nonmiss <- tibble(var = var) %>% filter(!is.na(var))
  total <- df_nonmiss %>%
    summarise(med = median(var), q1 = quantile(var, 0.25), q3 = quantile(var, 0.75)) %>%
    mutate(Total = sprintf("%.1f [%.1f\u2013%.1f]", med, q1, q3)) %>%
    dplyr::select(Total)
  n_miss <- sum(is.na(var))
  total$Missing <- sprintf("%d (%.1f%%)", n_miss, 100 * n_miss / length(var))
  total
}

# Categorical summary, total only (n (%) + Missing); yes/no collapses to the "yes" row
cat_summary_total <- function(var, varname = NULL) {
  if (is.null(varname)) varname <- "Variable"
  varname <- beautify_varname(varname)
  df_nonmiss <- tibble(value = var) %>% filter(!is.na(value))
  is_binary_yes_no <- length(unique(na.omit(var))) == 2 &&
    all(tolower(levels(factor(var))) %in% c("yes", "no"))
  total <- df_nonmiss %>%
    group_by(value) %>%
    summarise(n = n(), .groups = "drop") %>%
    mutate(p = n / sum(n), Total = sprintf("%d (%.1f%%)", n, p * 100))
  if (is_binary_yes_no) {
    total <- total %>% filter(tolower(value) == "yes") %>% mutate(Variable = varname)
  } else {
    total <- total %>% mutate(Variable = paste0(varname, ": ", value))
  }
  n_miss <- sum(is.na(var))
  total$Missing <- sprintf("%d (%.1f%%)", n_miss, 100 * n_miss / length(var))
  total %>% dplyr::select(Variable, Total, Missing)
}

create_descriptive_table_total <- function(df, var_spec) {
  map_dfr(seq_len(nrow(var_spec)), function(i) {
    v    <- var_spec$var[i] 
    type <- var_spec$type[i]
    if (type == "num") {
      tibble(Variable = beautify_varname(v)) %>% bind_cols(num_summary_total(df[[v]]))
    } else {
      cat_summary_total(df[[v]], varname = v)
    }
  }) %>%
    mutate(across(everything(), ~ replace_na(as.character(.), "-")))
}


var_spec_sepsis <- tribble(
  ~var,                          ~type,
  # clinical presentation and admission diagnoses
  "difficulty_breathing",        "cat",
  "fever",                       "cat",
  "convulsion",                  "cat",
  "early_onset_sepsis",          "cat",
  "congenital_pneumonia",        "cat",
  # inflammatory and haematological markers
  "c_reactive_protein_1",        "num",
  "leukocytes_1",                "num",
  "neutrophils_1",               "num",
  "platelets_1",                 "num",
  "hemoglobin_1",                "num",
  # microbiology
  "blood_culture_gram_stain",    "cat",
  "blood_culture_isolate",       "cat",
  "esbl",                        "cat",
  "stool_c1",                    "cat",
  # antibiotic exposure
  "antibiotics_perinatal",       "cat",
  "empirical_regimen",           "cat",
  "antibiotics_change_given",    "cat",
  # devices and outcome
  "nasogastric_tube",            "cat",
  "outcome",                     "cat",
  "hospital_stay",               "num"
)

table_sepsis <- create_descriptive_table_total(df = sepsis_only, var_spec = var_spec_sepsis)
table_sepsis
write.csv(table_sepsis[-3], file.path(out_dir, "descriptive_sepsis_characterisation.csv"), row.names = FALSE)

# blood-culture isolates listed separately (too many levels for the main table)
sepsis_only %>% count(blood_culture_isolate, name = "n") %>% arrange(desc(n)) %>% print(n = Inf)

In [ ]:
meta

# Analysis microbiome data

## Checks and filtering

### Import data (16S, ampliseq)

In [ ]:
theme_set(theme_light())
options(repr.plot.res=250)
expandedPalette = colorRampPalette(brewer.pal(9, "Paired"))

In [ ]:
phy_16S_raw  <- readRDS("../analysis/ampliseq_silva/phyloseq/dada2_phyloseq.rds")

In [ ]:
# standardise rank names to lowercase 
colnames(tax_table(phy_16S_raw)) <- tolower(colnames(tax_table(phy_16S_raw)))

rank_names(phy_16S_raw)

In [ ]:
sample_data(phy_16S_raw)

In [ ]:
# grouping variable: control = reference level, sepsis = comparison
sample_data(phy_16S_raw)$group <-
  factor(sample_data(phy_16S_raw)$group, levels = c("control", "sepsis", "mock"))

table(sample_data(phy_16S_raw)$group, useNA = "always")

### Filtered dataset

In [ ]:
# reads per sample (post-DADA2), sorted
tibble(sample = names(sample_sums(phy_16S_raw)),
       reads  = sample_sums(phy_16S_raw)) %>%
  arrange(reads)

In [ ]:
min_reads <- 5000   

phy_16S_raw_filtered <- prune_samples(sample_sums(phy_16S_raw) >= min_reads, phy_16S_raw)

# keep only the two biological groups (drops mock and any unlabelled/NA in one step)
keep <- sample_data(phy_16S_raw_filtered)$group %in% c("control", "sepsis")
phy_16S_raw_filtered <- prune_samples(keep, phy_16S_raw_filtered)

# set factor AFTER mock removal, control = reference; as.character drops the unused "mock" level
sample_data(phy_16S_raw_filtered)$group <-
  factor(as.character(sample_data(phy_16S_raw_filtered)$group), levels = c("control", "sepsis"))

phy_16S_raw_filtered

In [ ]:
# samples per group before and after filtering
table(sample_data(phy_16S_raw)$group)
table(sample_data(phy_16S_raw_filtered)$group)

### Fraction of reads classified at family / genus level

In [ ]:
is_classified <- function(x) !(is.na(x) | trimws(x) %in% c("", "NA", "unclassified"))
tax_tab_16S <- as.data.frame(tax_table(phy_16S_raw_filtered))
read_per_taxon <- taxa_sums(phy_16S_raw_filtered)
tibble(
  taxon = names(read_per_taxon),
  reads = read_per_taxon,
  family_classified = is_classified(tax_tab_16S$family),   
  genus_classified  = is_classified(tax_tab_16S$genus)     
) %>%
  summarise(
    total_reads = sum(reads),
    prop_family_asv   = mean(family_classified),                          
    prop_genus_asv    = mean(genus_classified),
    prop_family_reads = sum(reads[family_classified]) / total_reads,
    prop_genus_reads  = sum(reads[genus_classified])  / total_reads
  )

### Aggregate at genus level

In [ ]:
# keep only real taxonomic ranks, but ONLY those actually present
wanted <- c("domain","kingdom","phylum","class","order","family","genus","species")
ranks_keep <- intersect(wanted, colnames(tax_table(phy_16S_raw_filtered)))
tax_table(phy_16S_raw_filtered) <- tax_table(phy_16S_raw_filtered)[, ranks_keep]

# convert empty strings to NA
tt <- as.matrix(tax_table(phy_16S_raw_filtered))
tt[trimws(tt) %in% c("", "NA", "unclassified")] <- NA
tax_table(phy_16S_raw_filtered) <- tax_table(tt)


keep <- !is.na(as.data.frame(tax_table(phy_16S_raw_filtered))$family)
phy_16S_gen_filtered <- prune_taxa(keep, phy_16S_raw_filtered) %>%
  propagate_incomplete_taxonomy() %>%
  tax_glom("genus", NArm = FALSE) %>%
  prune_taxa(taxa_sums(.) > 0, .)

phy_16S_gen_filtered

In [ ]:
tt_gen <- as.data.frame(tax_table(phy_16S_gen_filtered))
tt_gen$genus[duplicated(tt_gen$genus)]

In [ ]:
# reads and ASVs removed by the family-level filter
dropped <- sum(!keep)
share   <- sum(taxa_sums(phy_16S_raw_filtered)[!keep]) / sum(taxa_sums(phy_16S_raw_filtered))
cat(sprintf("family filter: %d ASVs removed (%.3f%% of reads)\n", dropped, 100 * share))

In [ ]:
# Relative abundance
phy_16S_rel_gen_filtered <- transform_sample_counts(
  phy_16S_gen_filtered, function(x) x / sum(x)
)
phy_16S_rel_gen_filtered

In [ ]:
read_depth <- tibble(sample_id = sample_names(phy_16S_raw),
                     group = as.character(sample_data(phy_16S_raw)$group),
                     reads = sample_sums(phy_16S_raw)) %>%
  mutate(retained = reads >= min_reads & group %in% c("control", "sepsis")) %>%
  arrange(reads)
write.csv(read_depth, file.path(out_dir, "read_depth_per_sample.csv"), row.names = FALSE)
read_depth %>% count(group, retained)

## Explore positive control
Observed genus composition of the mock sample against the theoretical composition
of the ZymoBIOMICS Microbial Community Standard, expressed as 16S rRNA gene
abundance.

In [ ]:
mock_id <- sample_names(phy_16S_raw)[sample_data(phy_16S_raw)$group == "mock"]
stopifnot(length(mock_id) == 1)

phy_mock <- prune_samples(mock_id, phy_16S_raw)
phy_mock <- prune_taxa(taxa_sums(phy_mock) > 0, phy_mock)

cat("mock sample:", mock_id, "with", sum(sample_sums(phy_mock)), "reads and",
    ntaxa(phy_mock), "non-zero ASVs\n")

mock_asv <- psmelt(phy_mock) %>%
  as_tibble() %>%
  mutate(genus = if_else(is.na(genus) | genus == "", "Unclassified", as.character(genus)))

mock_obs <- mock_asv %>%
  group_by(genus) %>%
  summarise(reads = sum(Abundance), n_asv = n(), .groups = "drop") %>%
  mutate(observed = reads / sum(reads)) %>%
  arrange(desc(observed))

mock_obs %>% filter(observed >= 0.0001) %>% print(n = Inf)

# ASV level detail, useful to see whether V3-V4 separates the two Enterobacteriaceae
mock_asv %>%
  filter(Abundance > 0) %>%
  mutate(rel = Abundance / sum(Abundance)) %>%
  arrange(desc(rel)) %>%
  select(genus, species, species_exact, reads = Abundance, rel) %>%
  slice_head(n = 15) %>%
  print(n = Inf)


In [ ]:
# theoretical composition of ZymoBIOMICS D6300 in terms of 16S rRNA gene
# abundance. The two yeasts are not amplified by 16S.
# aliases cover SILVA 138 vs 138.1 naming, where Lactobacillus fermentum became Limosilactobacillus fermentum

mock_expected <- tibble::tribble(
  ~expected_species,          ~aliases,                        ~expected,
  "Pseudomonas aeruginosa",   "Pseudomonas",                    4.2,
  "Escherichia coli",         "Escherichia-Shigella,Escherichia", 10.1,
  "Salmonella enterica",      "Salmonella",                     10.4,
  "Lactobacillus fermentum",  "Limosilactobacillus,Lactobacillus", 18.4,
  "Enterococcus faecalis",    "Enterococcus",                   9.9,
  "Staphylococcus aureus",    "Staphylococcus",                 15.5,
  "Listeria monocytogenes",   "Listeria",                       14.1,
  "Bacillus subtilis",        "Bacillus",                       17.4
) %>%
  mutate(expected = expected / 100)

alias_lookup <- mock_expected %>%
  separate_longer_delim(aliases, delim = ",") %>%
  transmute(genus = str_trim(aliases), expected_species)

mock_cmp <- mock_obs %>%
  left_join(alias_lookup, by = "genus") %>%
  mutate(on_target = !is.na(expected_species)) %>%
  group_by(expected_species) %>%
  summarise(observed = sum(observed), matched_genera = paste(genus, collapse = ", "),
            .groups = "drop") %>%
  right_join(mock_expected %>% select(expected_species, expected),
             by = "expected_species") %>%
  mutate(observed = replace_na(observed, 0),
         log2_ratio = log2((observed + 1e-6) / expected))

on_target  <- sum(mock_cmp$observed)
off_target <- 1 - on_target
n_detected <- sum(mock_cmp$observed > 0)

cat(sprintf("expected taxa detected: %d of %d\n", n_detected, nrow(mock_cmp)))
cat(sprintf("on-target fraction:     %.3f\n", on_target))
cat(sprintf("off-target fraction:    %.3f\n", off_target))


mock_cmp %>% arrange(desc(expected)) %>% print(n = Inf)

# the off-target taxa are the false positives of the whole workflow
mock_off <- mock_obs %>%
  anti_join(alias_lookup, by = "genus") %>%
  arrange(desc(observed))
mock_off %>% slice_head(n = 15) %>% print(n = Inf)


write.csv(mock_cmp, file.path(out_dir, "mock_observed_vs_expected.csv"), row.names = FALSE)
write.csv(mock_off, file.path(out_dir, "mock_off_target_taxa.csv"), row.names = FALSE)

In [ ]:
mock_long <- mock_cmp %>%
  select(expected_species, Observed = observed, Expected = expected) %>%
  pivot_longer(c(Observed, Expected), names_to = "source", values_to = "rel") %>%
  mutate(source = factor(source, levels = c("Expected", "Observed")),
         expected_species = factor(expected_species,
                                   levels = rev(mock_cmp$expected_species[order(mock_cmp$expected)])))

p_mock <- ggplot(mock_long, aes(x = expected_species, y = rel, fill = source)) +
  geom_col(position = position_dodge(width = 0.75), width = 0.68) +
  scale_fill_manual(values = c(Expected = "grey65", Observed = "#2c6fbb"), name = NULL) +
  scale_y_continuous(labels = scales::percent_format(accuracy = 1), expand = expansion(c(0, 0.05))) +
  coord_flip() +
  labs(x = NULL, y = "Relative abundance (16S)") +
  theme(panel.grid.major.y = element_blank(), legend.position = "bottom")
p_mock



In [ ]:
# stacked bar 
# identity, everything else is pooled as off-target
mock_stack <- mock_obs %>%
  left_join(alias_lookup, by = "genus") %>%
  mutate(cat = if_else(is.na(expected_species), "Off-target", expected_species)) %>%
  group_by(cat) %>%
  summarise(observed = sum(observed), .groups = "drop") %>%
  bind_rows(mock_expected %>% transmute(cat = expected_species, expected)) %>%
  pivot_longer(c(observed, expected), names_to = "source", values_to = "rel") %>%
  filter(!is.na(rel)) %>%
  mutate(source = factor(if_else(source == "observed", "Observed", "Expected"),
                         levels = c("Expected", "Observed")))

cat_levels <- c(mock_expected$expected_species[order(-mock_expected$expected)], "Off-target")
pal_mock <- setNames(expandedPalette(length(cat_levels) - 1),
                     setdiff(cat_levels, "Off-target"))
pal_mock["Off-target"] <- "grey80"

p_mock_stack <- ggplot(mock_stack,
                       aes(x = source, y = rel, fill = factor(cat, levels = cat_levels))) +
  geom_col(position = "stack", width = 0.6) +
  scale_fill_manual(values = pal_mock, name = NULL) +
  scale_y_continuous(labels = scales::percent_format(accuracy = 1),
                     limits = c(0, 1), expand = c(0, 0)) +
  labs(x = NULL, y = "Relative abundance") +
  theme(panel.grid.major.x = element_blank())
p_mock_stack

ggsave(file.path(out_dir, "mock_barplot_stacked.tiff"), p_mock_stack,
       width = 12, height = 10, units = "cm",
       dpi = 600, compression = "lzw", bg = "white")

ggsave(file.path(out_dir, "mock_barplot.pdf"), p_mock_stack,
       width = 14, height = 9, units = "cm")
ggsave(file.path(out_dir, "mock_barplot.tiff"), p_mock_stack,
       width = 14, height = 9, units = "cm",
       dpi = 600, compression = "lzw", bg = "white")       

## Barplots

In [ ]:
# Composition barplot, genus level
# Sepsis samples are ordered by stratum, pathogen genus and species, then by the
# abundance of their own blood culture genus; controls by Bray-Curtis clustering.
# Sepsis labels carry the blood culture isolate.

n_top   <- 15
id_mode <- "sequential"   # "sequential" = P01/H01 numbering, "keep" = original ids

# figure geometry in mm. 180 = full text width of most journals, 85 = single
# column. Sizes are absolute so the panel can be combined with others later
# without rescaling.
fig_width_mm  <- 180
fig_height_mm <- 95
base_family   <- ""       # e.g. "Helvetica" or "Arial"
size_base     <- 8        # pt, body text
size_axis     <- 6        # pt, sample labels
size_legend   <- 6.5      # pt

map_bc <- tibble::tribble(
  ~organism_key,           ~silva_genus,           ~stratum,
  "acinetobacterspp",      "Acinetobacter",        "non-Enterobacterales",
  "kpneumoniae",           "Klebsiella",           "Enterobacterales",
  "cons",                  "Staphylococcus",       "non-Enterobacterales",
  "saureus",               "Staphylococcus",       "non-Enterobacterales",
  "ecoli",                 "Escherichia-Shigella", "Enterobacterales",
  "kaerogenes",            "Klebsiella",           "Enterobacterales",
  "paeruginosa",           "Pseudomonas",          "non-Enterobacterales",
  "streptococcusspp",      "Streptococcus",        "non-Enterobacterales"
)

# sample x genus matrix 
comp_ab <- as(otu_table(phy_16S_rel_gen_filtered), "matrix")
if (taxa_are_rows(phy_16S_rel_gen_filtered)) comp_ab <- t(comp_ab)
colnames(comp_ab) <- as.character(tax_table(phy_16S_rel_gen_filtered)[, "genus"])
stopifnot(!any(duplicated(colnames(comp_ab))))

comp_long <- as_tibble(comp_ab, rownames = "sample_id") %>%
  pivot_longer(-sample_id, names_to = "taxon", values_to = "rel")

# group per sample, blood culture isolate joined from meta.
comp_smp <- tibble(sample_id = sample_names(phy_16S_rel_gen_filtered),
                   grp = as.character(sample_data(phy_16S_rel_gen_filtered)$group)) %>%
  left_join(
    meta %>%
      transmute(sample_id = str_remove(str_trim(as.character(study_id)), "-P$"),
                organism  = as.character(blood_culture_isolate),
                key       = str_replace_all(str_to_lower(organism), "[^a-z]", "")),
    by = "sample_id") %>%
  left_join(map_bc %>% select(key = organism_key, silva_genus, stratum), by = "key") %>%
  mutate(own = comp_ab[cbind(match(sample_id, rownames(comp_ab)),
                             match(silva_genus, colnames(comp_ab)))])

# display ids using sequential numbering
comp_smp <- comp_smp %>%
  arrange(grp, sample_id) %>%
  group_by(grp) %>%
  mutate(display = if (id_mode == "keep") sample_id
                   else sprintf("%s%02d", if (first(grp) == "sepsis") "P" else "H",
                                row_number())) %>%
  ungroup() %>%
  mutate(label = if_else(grp == "sepsis" & !is.na(organism) & organism != "",
                         paste0(display, " \u00b7 ", organism), display))

stopifnot(!any(duplicated(comp_smp$display)))

write.csv(comp_smp %>% select(sample_id, grp, display, organism, silva_genus, stratum),
          file.path(out_dir, "sample_id_crosswalk.csv"), row.names = FALSE)

# sepsis by pathogen block, controls by average linkage on Bray-Curtis
comp_ctrl <- comp_smp$sample_id[comp_smp$grp == "control"]
comp_ord  <- c(
  comp_smp %>% filter(grp == "sepsis") %>%
    mutate(across(c(stratum, silva_genus, organism), ~ replace_na(.x, "zz")),
           own = replace_na(own, -1)) %>%
    arrange(stratum, silva_genus, organism, desc(own)) %>% pull(sample_id),
  comp_ctrl[hclust(vegan::vegdist(comp_ab[comp_ctrl, , drop = FALSE], "bray"),
                   "average")$order]
)
stopifnot(setequal(comp_ord, comp_smp$sample_id))

comp_top <- sort(names(sort(colMeans(comp_ab), decreasing = TRUE))[seq_len(n_top)])

comp_df <- comp_long %>%
  mutate(taxon = if_else(taxon %in% comp_top, taxon, "Other")) %>%
  group_by(sample_id, taxon) %>% summarise(rel = sum(rel), .groups = "drop") %>%
  left_join(comp_smp %>% select(sample_id, grp, label), by = "sample_id") %>%
  mutate(sample_id = factor(sample_id, levels = comp_ord),
         label     = fct_reorder(label, as.integer(sample_id)),
         taxon     = factor(taxon, levels = c(comp_top, "Other")),
         grp       = factor(grp, levels = c("sepsis", "control"),
                            labels = c(lab_bsi, lab_healthy)))

comp_pal <- setNames(
  colorRampPalette(RColorBrewer::brewer.pal(12, "Paired"))(length(comp_top)), comp_top)
comp_pal["Other"] <- "grey80"

# theme with absolute point sizes
theme_pub <- theme_minimal(base_size = size_base, base_family = base_family) +
  theme(
    axis.text.x      = element_text(angle = 90, hjust = 1, vjust = 0.5, size = size_axis),
    axis.text.y      = element_text(size = size_axis),
    axis.title       = element_text(size = size_base),
    axis.line.y      = element_line(linewidth = 0.3, colour = "black"),
    axis.ticks.y     = element_line(linewidth = 0.3, colour = "black"),
    panel.grid       = element_blank(),
    panel.spacing    = unit(2, "mm"),
    strip.text       = element_text(size = size_base, face = "bold", margin = margin(b = 1.5, unit = "mm")),
    legend.text      = element_text(size = size_legend),
    legend.title     = element_blank(),
    legend.key.size  = unit(3, "mm"),
    legend.margin    = margin(0, 0, 0, 0),
    legend.box.spacing = unit(2, "mm"),
    plot.margin      = margin(1, 1, 1, 1, unit = "mm"),
    plot.tag         = element_text(size = size_base + 2, face = "bold")
  )

p_comp <- ggplot(comp_df, aes(x = label, y = rel, fill = taxon)) +
  geom_col(width = 0.85) +
  facet_grid(~ grp, scales = "free_x", space = "free_x") +
  scale_fill_manual(values = comp_pal) +
  scale_y_continuous(labels = scales::percent_format(accuracy = 1),
                     limits = c(0, 1), expand = c(0, 0)) +
  guides(fill = guide_legend(ncol = 1, byrow = TRUE)) +
  labs(x = "Neonate (neonates with BSI labeled with blood culture isolate)",
       y = "Relative abundance") +
  theme_pub

# vector for the composite figure, raster at 600 dpi for submission
ggsave(file.path(out_dir, "figure1_composition_genus_by_pathogen.pdf"), p_comp,
       width = fig_width_mm, height = fig_height_mm, units = "mm",
       device = cairo_pdf)
ggsave(file.path(out_dir, "figure1_composition_genus_by_pathogen.tiff"), p_comp,
       width = fig_width_mm, height = fig_height_mm, units = "mm",
       dpi = 600, compression = "lzw", bg = "white")

# preview at the true output size inside the notebook
options(repr.plot.width = fig_width_mm / 25.4, repr.plot.height = fig_height_mm / 25.4)
p_comp

In [ ]:
# Dominant genus per sample and dominance at the 30 percent threshold
dom_threshold <- 0.30


comp_rel_long <- as_tibble(comp_ab, rownames = "sample_id") %>%
  pivot_longer(-sample_id, names_to = "genus", values_to = "rel") %>%
  left_join(comp_smp %>% select(sample_id, grp, display, organism), by = "sample_id")

dominant_genus <- comp_rel_long %>%
  group_by(sample_id) %>%
  slice_max(rel, n = 1, with_ties = TRUE) %>%
  ungroup()

dom_ties <- dominant_genus %>% count(sample_id) %>% filter(n > 1)
if (nrow(dom_ties)) stop("tied most abundant genus in: ", toString(dom_ties$sample_id))

dominant_genus <- dominant_genus %>%
  mutate(dominated = rel >= dom_threshold) %>%
  select(sample_id, display, grp, organism,
         dominant_genus = genus, rel_abundance = rel, dominated) %>%
  arrange(grp, desc(rel_abundance))

stopifnot(nrow(dominant_genus) == nsamples(phy_16S_rel_gen_filtered))

write.csv(dominant_genus, file.path(out_dir, "dominant_genus_per_sample.csv"),
          row.names = FALSE)

# how often each genus dominates a community, by group
dominance_by_group <- dominant_genus %>%
  filter(dominated) %>%
  count(grp, dominant_genus, name = "n_dominated") %>%
  arrange(grp, desc(n_dominated))

# per genus and group: highest relative abundance and how often it reaches the threshold
genus_peak <- comp_rel_long %>%
  group_by(grp, genus) %>%
  summarise(max_rel        = max(rel),
            n_at_threshold = sum(rel >= dom_threshold), .groups = "drop") %>%
  filter(max_rel > 0) %>%
  arrange(genus, grp)

write.csv(genus_peak, file.path(out_dir, "genus_peak_abundance_by_group.csv"),
          row.names = FALSE)

print(dominance_by_group, n = Inf)

dominant_genus %>%
  mutate(enterobacterales = dominated &
           dominant_genus %in% c("Klebsiella", "Escherichia-Shigella")) %>%
  count(grp, enterobacterales)

## Diversity with Divnet

In [ ]:
# Prevalence per taxon — helps judge a stable DivNet base if the default is unstable
prevalence_16S <- apply(otu_table(phy_16S_gen_filtered), 1, function(x) sum(x > 0))
tax_16S        <- as.data.frame(tax_table(phy_16S_gen_filtered))
n_samples_16S  <- nsamples(phy_16S_gen_filtered)

data.frame(
  taxon_id          = names(prevalence_16S),
  n_samples_present = prevalence_16S,
  pct_samples       = round(100 * prevalence_16S / n_samples_16S, 1),
  total_reads       = taxa_sums(phy_16S_gen_filtered),
  genus             = tax_16S$genus
) %>%
  arrange(desc(n_samples_present), desc(total_reads)) %>%
  head(15)

### Alpha-Diversity

In [ ]:
# DivNet fit with identity design: one estimate and standard error per sample
X_alpha <- diag(nsamples(phy_16S_gen_filtered))
rownames(X_alpha) <- colnames(X_alpha) <- sample_names(phy_16S_gen_filtered)

set.seed(123)
divnet_16S_alpha <- divnet(phy_16S_gen_filtered, X = X_alpha,
                           ncores = 1, tuning = "careful")

In [ ]:
# per-sample estimates and standard errors in long format
lab_simpson <- "Simpson (\u03a3p\u1d62\u00b2)"

get_div <- function(div_idx, what) {
  vapply(div_idx, function(x) unname(as.numeric(x[[what]])), numeric(1))
}

meta_alpha <- as(sample_data(phy_16S_gen_filtered), "data.frame") %>%
  rownames_to_column("sample_id") %>%
  mutate(group = factor(as.character(group), levels = c("control", "sepsis")))

alpha_ps <- bind_rows(
  tibble(index     = "Shannon",
         sample_id = names(divnet_16S_alpha$shannon),
         estimate  = get_div(divnet_16S_alpha$shannon, "estimate"),
         se        = get_div(divnet_16S_alpha$shannon, "error")),
  tibble(index     = lab_simpson,
         sample_id = names(divnet_16S_alpha$simpson),
         estimate  = get_div(divnet_16S_alpha$simpson, "estimate"),
         se        = get_div(divnet_16S_alpha$simpson, "error"))
) %>%
  left_join(select(meta_alpha, sample_id, group), by = "sample_id")

# every sample must be matched to a group, once per index
stopifnot(!any(is.na(alpha_ps$group)),
          nrow(alpha_ps) == 2 * nsamples(phy_16S_gen_filtered))

alpha_ps %>% group_by(index, group) %>%
  summarise(n = n(), median_est = median(estimate), median_se = median(se), .groups = "drop")

In [ ]:
# betta per index: group effect accounting for per-sample measurement error
# and heterogeneity between samples; group means via betta_lincom
fit_betta_index <- function(df) {
  bt   <- breakaway::betta(formula = estimate ~ group, ses = se, data = df)
  tab  <- bt$table
  ctrl <- breakaway::betta_lincom(bt, linear_com = c(1, 0), signif_cutoff = 0.05)
  bsi  <- breakaway::betta_lincom(bt, linear_com = c(1, 1), signif_cutoff = 0.05)
  z    <- qnorm(0.975)
  list(
    fit    = bt,
    groups = tibble(group = factor(c("control", "sepsis"), levels = c("control", "sepsis")),
                    mean  = c(ctrl[1, "Estimates"], bsi[1, "Estimates"]),
                    lower = c(ctrl[1, "Lower CIs"], bsi[1, "Lower CIs"]),
                    upper = c(ctrl[1, "Upper CIs"], bsi[1, "Upper CIs"])),
    effect = tibble(diff  = tab[2, "Estimates"],
                    se    = tab[2, "Standard Errors"],
                    lower = tab[2, "Estimates"] - z * tab[2, "Standard Errors"],
                    upper = tab[2, "Estimates"] + z * tab[2, "Standard Errors"],
                    p     = tab[2, "p-values"],
                    ssq_u = if (!is.null(bt$ssq_u)) bt$ssq_u else NA_real_)
  )
}

betta_alpha   <- split(alpha_ps, alpha_ps$index) %>% lapply(fit_betta_index)
alpha_effects <- bind_rows(lapply(betta_alpha, `[[`, "effect"), .id = "index")
alpha_groups  <- bind_rows(lapply(betta_alpha, `[[`, "groups"), .id = "index")

lapply(betta_alpha, function(x) x$fit$table)
alpha_effects
alpha_groups

pvals_16s <- setNames(alpha_effects$p, alpha_effects$index)

write.csv(alpha_ps,      file.path(out_dir, "alpha_divnet_per_sample.csv"), row.names = FALSE)
write.csv(alpha_effects, file.path(out_dir, "alpha_betta_effects.csv"),      row.names = FALSE)
write.csv(alpha_groups,  file.path(out_dir, "alpha_betta_group_means.csv"),  row.names = FALSE)

In [ ]:
# per-sample estimates (jittered), betta group means with 95% CI
plot_alpha_samples <- function(alpha_ps, alpha_groups, group_colors, pvals = NULL,
                               base_size = 8, y_label = "Estimate", title = NULL) {
    pd <- position_jitter(width = 0.12, height = 0, seed = 123)
  relab <- function(x) factor(recode(as.character(x), control = lab_healthy, sepsis = lab_bsi),
                              levels = c(lab_bsi, lab_healthy))

  df <- alpha_ps %>% mutate(group = relab(group))
  gm <- alpha_groups %>% mutate(group = relab(group))

  p <- ggplot(df, aes(x = group, y = estimate, color = group)) +
    geom_point(position = pd, size = 1.2, alpha = 0.6) +
    geom_errorbar(data = gm, aes(x = group, ymin = lower, ymax = upper),
                  inherit.aes = FALSE, width = 0.25, linewidth = 0.6, color = "black") +
    geom_point(data = gm, aes(x = group, y = mean), inherit.aes = FALSE,
               shape = 23, size = 2.4, fill = "white", color = "black") +
    facet_wrap(~ index, scales = "free_y") +
    scale_color_manual(values = group_colors) +
    scale_x_discrete(labels = function(x) stringr::str_wrap(x, 10)) +
    labs(x = NULL, y = y_label, title = title) +
    theme_minimal(base_size = base_size) +
    theme(legend.position = "none", panel.grid.minor = element_blank(),
          plot.title = element_text(face = "bold"),
          strip.text = element_text(face = "bold"))

  if (!is.null(pvals)) {
    fmt_p <- function(p) if (p < 0.001) "italic(p) < 0.001" else sprintf("italic(p) == %.3f", p)
    pval_df <- tibble(index = names(pvals), label = vapply(pvals, fmt_p, character(1)))
    p <- p + geom_text(data = pval_df, aes(x = -Inf, y = Inf, label = label),
                       hjust = -0.1, vjust = 1.5, inherit.aes = FALSE, parse = TRUE,
                       size = base_size * 0.3)
  }
  p
}

group_colors_pc <- setNames(c("#45458b", "#056b05"), c(lab_bsi, lab_healthy))

plot_16s_alpha <- plot_alpha_samples(alpha_ps, alpha_groups, group_colors_pc,
                                     pvals = pvals_16s, title = "16S alpha diversity (genus)")

ggsave(file.path(out_dir, "alpha_16S.pdf"), plot_16s_alpha,
       width = 8.5, height = 6, units = "cm", device = cairo_pdf)
ggsave(file.path(out_dir, "alpha_16S.tiff"), plot_16s_alpha,
       width = 8.5, height = 6, units = "cm", dpi = 600, compression = "lzw", bg = "white")
plot_16s_alpha

### Beta diversity (Bray-Curtis, DivNet)

#### Beta diversity at genus level

In [ ]:
# Bray-Curtis test on the same per-sample DivNet fit used for alpha diversity
set.seed(123)
bc_test_16S_pc <- testBetaDiversity(divnet_16S_alpha, h0 = "bray-curtis",
                                    n_boot = 10000, sample_specimen_matrix = X_alpha,
                                    groups = sample_data(phy_16S_gen_filtered)$group)
bc_test_16S_pc$p_value

In [ ]:
# PCoA on the DivNet Bray-Curtis matrix
bc_mat <- divnet_16S_alpha[["bray-curtis"]]
bc_mat[bc_mat < 0] <- 0
diag(bc_mat) <- 0
bc_dist <- as.dist(bc_mat)

pcoa_res <- ape::pcoa(bc_dist, correction = "cailliez")
eig_col  <- if ("Rel_corr_eig" %in% colnames(pcoa_res$values)) "Rel_corr_eig" else "Relative_eig"
var_expl <- round(100 * pcoa_res$values[[eig_col]][1:2], 1)

meta_pc <- as(sample_data(phy_16S_gen_filtered), "data.frame") %>% rownames_to_column("sample_id")
pcoa_df <- as.data.frame(pcoa_res$vectors[, 1:2]) %>%
  rownames_to_column("sample_id") %>%
  rename(Axis1 = Axis.1, Axis2 = Axis.2) %>%
  left_join(meta_pc, by = "sample_id") %>%
  mutate(group = recode(as.character(group), control = lab_healthy, sepsis = lab_bsi),
         group = factor(group, levels = c(lab_bsi, lab_healthy)))

p_lab <- if (bc_test_16S_pc$p_value < 0.001) "italic(p) < 0.001" else
         sprintf("italic(p) == %.3f", bc_test_16S_pc$p_value)

plot_16s_pcoa <- ggplot(pcoa_df, aes(Axis1, Axis2, color = group, fill = group)) +
  stat_ellipse(geom = "polygon", alpha = 0.15, linewidth = 0.4, level = 0.95) +
  geom_point(size = 2) +
  scale_color_manual(values = group_colors_pc) +
  scale_fill_manual(values = group_colors_pc) +
  labs(x = sprintf("PCoA 1 (%.1f%%)", var_expl[1]),
       y = sprintf("PCoA 2 (%.1f%%)", var_expl[2]), color = NULL, fill = NULL,
       title = "Beta diversity (genus)") +     
  annotate("text", x = -Inf, y = Inf, label = p_lab, hjust = 0, vjust = 1,
           parse = TRUE, size = 8 * 0.36) +
  theme_minimal(base_size = 8) +
  theme(legend.position = "bottom", panel.grid.minor = element_blank(),
        plot.title = element_text(face = "bold"))

ggsave(file.path(out_dir, "pcoa_16S.pdf"),  plot_16s_pcoa, width = 8.5, height = 8.5, units = "cm")
ggsave(file.path(out_dir, "pcoa_16S.tiff"), plot_16s_pcoa, width = 8.5, height = 8.5, units = "cm",
       dpi = 600, compression = "lzw")
plot_16s_pcoa

In [ ]:
# combined diversity figure: alpha and beta as panels A and B
fig2 <- (plot_16s_alpha + labs(title = NULL) | plot_16s_pcoa + labs(title = NULL)) +
  plot_layout(widths = c(1, 1)) +
  plot_annotation(tag_levels = "A") &
  theme(plot.tag = element_text(size = 10, face = "bold"))

ggsave(file.path(out_dir, "figure2_diversity.pdf"), fig2,
       width = 180, height = 80, units = "mm", device = cairo_pdf)
ggsave(file.path(out_dir, "figure2_diversity.tiff"), fig2,
       width = 180, height = 80, units = "mm", dpi = 600,
       compression = "lzw", bg = "white")
fig2

## Differential abundance with radEMU

In [ ]:
filter_nonzero_taxa <- function(physeq) {
  keep <- taxa_sums(physeq) > 0
  prune_taxa(keep, physeq)
}

fit_emu <- function(physeq, group_var) {
  taxa_to_test <- rownames(otu_table(physeq))
  emuFit(
    formula         = as.formula(paste0("~ ", group_var)),
    Y               = physeq,
    run_score_tests = TRUE,
    test_kj         = data.frame(k = 2, j = taxa_to_test),
    verbose         = TRUE
  )
}

In [ ]:
# Reference level control -> estimate is log-fold-change Sepsis vs Controls
sample_data(phy_16S_gen_filtered)$group <-
  factor(sample_data(phy_16S_gen_filtered)$group, levels = c("control", "sepsis"))

# Drop zero-only taxa
phy_16S_pc_emu <- filter_nonzero_taxa(phy_16S_gen_filtered)
cat("Taxa retained (sepsis vs controls):", ntaxa(phy_16S_pc_emu), "\n")

# run
emu_fit_16S_pc <- fit_emu(phy_16S_pc_emu, group_var = "group")

In [ ]:
dplyr::count(emu_fit_16S_pc$coef, covariate)

In [ ]:
tidy_emu <- function(fit, physeq, rank_col, prev_cutoff = 5) {

  # Multiple-testing correction settings
  adjust_method     <- "BH"
  sig_threshold_raw <- 0.05
  sig_threshold_adj <- 0.1

  res <- fit$coef %>%
    dplyr::select(category, estimate, se, lower, upper, pval)

  tax <- as.data.frame(tax_table(physeq))
  tax$category <- rownames(tax)
  res <- dplyr::left_join(res, tax, by = "category")

  res <- res %>%
    dplyr::mutate(
      label_raw = ifelse(is.na(.data[[rank_col]]), "Unclassified",
                         as.character(.data[[rank_col]]))
    ) %>%
    dplyr::group_by(label_raw) %>%
    dplyr::mutate(label = if (dplyr::n() > 1)
                    paste0(label_raw, "_", dplyr::row_number()) else label_raw) %>%
    dplyr::ungroup()

  otu_counts <- otu_table(physeq)
  if (taxa_are_rows(physeq)) otu_counts <- t(otu_counts)
  non_zero <- data.frame(
    category              = colnames(otu_counts),
    non_zero_sample_count = as.integer(colSums(otu_counts > 0))
  )
  res <- dplyr::left_join(res, non_zero, by = "category")

  res <- res %>%
    dplyr::mutate(
      qval    = p.adjust(pval, method = adjust_method),
      sig_adj = qval <= sig_threshold_adj,
      sig_raw = pval <= sig_threshold_raw,
      significant = dplyr::case_when(
        sig_adj            ~ "significant (adjusted)",
        sig_raw & !sig_adj ~ "significant",
        TRUE               ~ "not significant"
      ),
      color_group = dplyr::case_when(
        non_zero_sample_count < prev_cutoff & significant == "not significant"        ~ "not significant, few positive samples",
        non_zero_sample_count < prev_cutoff & significant == "significant"            ~ "significant, few positive samples",
        non_zero_sample_count < prev_cutoff & significant == "significant (adjusted)" ~ "significant (adjusted), few positive samples",
        significant == "significant (adjusted)"                                       ~ "significant (adjusted)",
        significant == "significant"                                                  ~ "significant",
        TRUE                                                                          ~ "not significant"
      ),
      label = factor(label, levels = unique(label[order(estimate)])),
      fold = exp(estimate), fold_lower = exp(lower), fold_upper = exp(upper)
    ) %>%
    dplyr::select(-sig_adj, -sig_raw)

  res
}

In [ ]:
# Build tidy result tables
emu_16S_pc_res <- tidy_emu(emu_fit_16S_pc, phy_16S_pc_emu, rank_col = "genus")

#save
write.csv(emu_16S_pc_res, file.path(out_dir, "radEmu_16S.csv"), row.names = FALSE)

emu_16S_pc_res %>%
dplyr::arrange(pval)%>% head


In [ ]:
plot_radEmu <- function(res, x_label,
                        sig_adj_col         = "#4b0808",
                        sig_raw_col         = "red",
                        nonsig_col          = "black",
                        sig_adj_lowprev_col = scales::alpha("#4b0808", 0.4),
                        sig_raw_lowprev_col = scales::alpha("red", 0.4),
                        nonsig_lowprev_col  = "grey70",
                        base_size           = 7) {

  color_vals <- c(
    "significant (adjusted)"                       = sig_adj_col,
    "significant"                                  = sig_raw_col,
    "not significant"                              = nonsig_col,
    "significant (adjusted), few positive samples" = sig_adj_lowprev_col,
    "significant, few positive samples"            = sig_raw_lowprev_col,
    "not significant, few positive samples"        = nonsig_lowprev_col
  )
  color_labs <- c(
    "significant (adjusted)"                       = "Significant\n(after correction)",
     "significant"                                  = "p \u2264 0.05 (unadjusted)",
    "not significant"                              = "Not significant",
    "significant (adjusted), few positive samples" = "Significant (corrected)\n(few positive samples)",
    "significant, few positive samples"            = "p \u2264 0.05 (unadjusted)\n(few positive samples)",
    "not significant, few positive samples"        = "Not significant\n(few positive samples)"
  )

  ggplot(res, aes(x = estimate, y = label, color = color_group)) +
    geom_vline(xintercept = 0, linetype = "dashed", color = "grey40") +
    geom_errorbarh(aes(xmin = lower, xmax = upper), height = 0.3) +
    geom_point(size = 1.2) +
    scale_color_manual(values = color_vals, labels = color_labs, name = NULL) +
    labs(x = x_label, y = NULL) +
    theme_minimal(base_size = base_size) +
    theme(
      panel.grid.major.y = element_blank(),
      legend.position    = "bottom",
      legend.key.size    = unit(0.3, "cm")
    )
}

In [ ]:
# Save helper: PDF + TIFF, height scales with taxa count
save_radEmu_plot <- function(plot, file_stub, n_taxa,
                             width_cm = 8.5, per_taxon_cm = 0.18, min_h = 8) {
  h <- max(min_h, n_taxa * per_taxon_cm)
  ggsave(paste0(file_stub, ".pdf"),  plot,
         width = width_cm, height = h, units = "cm")
  ggsave(paste0(file_stub, ".tiff"), plot,
         width = width_cm, height = h, units = "cm",
         dpi = 600, compression = "lzw", bg = "white")
}

In [ ]:
# Plot Sepsis vs Controls
p_16S_pc <- plot_radEmu(emu_16S_pc_res, x_label = "Log fold-difference (ln), BSI vs healthy")


save_radEmu_plot(p_16S_pc, file.path(out_dir, "radEmu_16S"), n_taxa = nrow(emu_16S_pc_res))
p_16S_pc

In [ ]:
plot_radEmu_sig_only <- function(res, x_label,
                                 sig_adj_col = "#4b0808",
                                 sig_raw_col = "red",
                                 base_size   = 7) {

  # Keep only significant categories (adjusted or raw), drop non-significant and low-prevalence
  keep_groups <- c("significant (adjusted)",
                   "significant")
  res_sig <- res %>%
    dplyr::filter(color_group %in% keep_groups) %>%
    dplyr::mutate(label = factor(label, levels = unique(label[order(estimate)])))

  if (nrow(res_sig) == 0) {
    message("No significant taxa to plot.")
    return(invisible(NULL))
  }

  color_vals <- c(
    "significant (adjusted)" = sig_adj_col,
    "significant"            = sig_raw_col
  )
  color_labs <- c(
    "significant (adjusted)" = "Significant\n(after correction)",
    "significant"            = "p \u2264 0.05 (unadjusted)"
  )

  ggplot(res_sig, aes(x = estimate, y = label, color = color_group)) +
    geom_vline(xintercept = 0, linetype = "dashed", color = "grey40") +
    geom_errorbarh(aes(xmin = lower, xmax = upper), height = 0.3) +
    geom_point(size = 1.2) +
    scale_color_manual(values = color_vals, labels = color_labs, name = NULL) +
    labs(x = x_label, y = NULL) +
    theme_minimal(base_size = base_size) +
    theme(
      panel.grid.major.y = element_blank(),
      legend.position    = "bottom",
      legend.key.size    = unit(0.3, "cm")
    )
}

In [ ]:
p_16S_pc_sig <- plot_radEmu_sig_only(emu_16S_pc_res, x_label = "Log fold-difference (ln), BSI vs healthy")

n_16S_sig <- sum(emu_16S_pc_res$color_group %in% c("significant (adjusted)", "significant"))
if (!is.null(p_16S_pc_sig))
  save_radEmu_plot(p_16S_pc_sig, file.path(out_dir, "radEmu_16S_SvC_sigOnly"), n_taxa = max(1, n_16S_sig))
  
p_16S_pc_sig

### Differential abundance adjusted for antibiotics during pregnancy

In [ ]:
# sample names correspond to study_id with the trailing "-P" removed
abx_preg <- meta %>%
  transmute(
    sample_id = str_remove(str_trim(as.character(study_id)), "-P$"),
    antibiotics_during_pregnancy = factor(as.character(antibiotics_during_pregnancy),
                                          levels = c("No", "Yes"))
  ) %>%
  distinct(sample_id, .keep_all = TRUE)

add_abx_preg <- function(physeq) {
  sd <- as(sample_data(physeq), "data.frame") %>% rownames_to_column("sample_id")
  sd <- dplyr::left_join(sd, abx_preg, by = "sample_id")
  rownames(sd) <- sd$sample_id
  sample_data(physeq) <- sample_data(sd[, setdiff(names(sd), "sample_id"), drop = FALSE])
  physeq
}

In [ ]:
# genus level: attach covariate, report unmatched samples, drop rows the model cannot use
phy_16S_pc_abx <- add_abx_preg(phy_16S_gen_filtered)

sd_chk <- as(sample_data(phy_16S_pc_abx), "data.frame")
cat("samples without covariate value:",
    paste(rownames(sd_chk)[is.na(sd_chk$antibiotics_during_pregnancy)], collapse = ", "), "\n")
print(table(sd_chk$group, sd_chk$antibiotics_during_pregnancy, useNA = "ifany"))

phy_16S_pc_abx <- prune_samples(
  !is.na(sample_data(phy_16S_pc_abx)$antibiotics_during_pregnancy), phy_16S_pc_abx)

# control stays the reference level, so the group coefficient is log-fold-change Sepsis vs Controls
sample_data(phy_16S_pc_abx)$group <-
  factor(as.character(sample_data(phy_16S_pc_abx)$group), levels = c("control", "sepsis"))

phy_16S_pc_abx <- filter_nonzero_taxa(phy_16S_pc_abx)
cat("samples retained:", nsamples(phy_16S_pc_abx),
    "| taxa retained:", ntaxa(phy_16S_pc_abx), "\n")

In [ ]:
n_missing <- sum(is.na(sd_chk$antibiotics_during_pregnancy))
stopifnot("too many samples without covariate value" = n_missing <= 1)

In [ ]:
# score tests are run for the group coefficient only; its position in the design matrix is looked up
# instead of hard-coding k = 2, so the code stays correct if the formula changes
fit_emu_adj <- function(physeq, formula_chr, test_covariate) {
  sd <- as(sample_data(physeq), "data.frame")
  X  <- model.matrix(as.formula(formula_chr), data = sd)
  k  <- match(test_covariate, colnames(X))
  if (is.na(k)) stop("covariate not in design matrix, available: ", toString(colnames(X)))
  emuFit(
    formula         = as.formula(formula_chr),
    Y               = physeq,
    run_score_tests = TRUE,
    test_kj         = data.frame(k = k, j = rownames(otu_table(physeq))),
    verbose         = TRUE
  )
}

In [ ]:
# slow - run once
emu_fit_16S_pc_abx <- fit_emu_adj(phy_16S_pc_abx,
                                  "~ group + antibiotics_during_pregnancy",
                                  "groupsepsis")

dplyr::count(emu_fit_16S_pc_abx$coef, covariate)

In [ ]:
# tidy_emu expects one row per category, so restrict coef to the group contrast
emu_fit_16S_pc_abx_group <- emu_fit_16S_pc_abx
emu_fit_16S_pc_abx_group$coef <-
  dplyr::filter(emu_fit_16S_pc_abx$coef, covariate == "groupsepsis")

emu_16S_pc_abx_res <- tidy_emu(emu_fit_16S_pc_abx_group, phy_16S_pc_abx, rank_col = "genus")

write.csv(emu_16S_pc_abx_res, file.path(out_dir, "radEmu_16S_abxPreg.csv"), row.names = FALSE)

emu_16S_pc_abx_res %>% dplyr::arrange(pval) %>% head()

In [ ]:
p_16S_pc_abx <- plot_radEmu(
  emu_16S_pc_abx_res,
  x_label = "Log fold-difference (ln), BSI vs healthy, adjusted for antibiotics during pregnancy")

save_radEmu_plot(p_16S_pc_abx, file.path(out_dir, "radEmu_16S_abxPreg"),
                 n_taxa = nrow(emu_16S_pc_abx_res))
p_16S_pc_abx

p_16S_pc_abx_sig <- plot_radEmu_sig_only(
  emu_16S_pc_abx_res,
  x_label = "Log fold-difference (ln), BSI vs healthy, adjusted for antibiotics during pregnancy")

n_16S_abx_sig <- sum(emu_16S_pc_abx_res$color_group %in%
                       c("significant (adjusted)", "significant"))
if (!is.null(p_16S_pc_abx_sig))
  save_radEmu_plot(p_16S_pc_abx_sig, file.path(out_dir, "radEmu_16S_abxPreg_sigOnly"),
                   n_taxa = max(1, n_16S_abx_sig))
p_16S_pc_abx_sig

saveRDS(emu_fit_16S_pc_abx, file.path(out_dir, "emuFit_16S_abxPreg.rds"))

In [ ]:
nrow(emu_16S_pc_res) 
nrow(emu_16S_pc_abx_res)

In [ ]:
# compare results of sensitivity analysis
# absolute shift, and what happens to the taxa that actually matter
inner_join(
  emu_16S_pc_res     %>% select(label, est_crude = estimate, p_crude = pval, q_crude = qval),
  emu_16S_pc_abx_res %>% select(label, est_adj   = estimate, p_adj   = pval, q_adj   = qval),
  by = "label") %>%
  mutate(abs_change = est_adj - est_crude) %>%
  filter(p_crude <= 0.1 | p_adj <= 0.1) %>%
  arrange(p_crude) %>%
  print(n = Inf)

# Blood cultue isolates

## Gut-blood concordance in microbiome (matched-taxon analysis)

### Create the datasets

In [ ]:
#strata definition for analysis, CoNS is a common contaminant and therefore flagged
bc_map <- tibble::tribble(
  ~organism,             ~silva_genus,           ~stratum,             ~is_cons,
  "K. pneumoniae",        "Klebsiella",           "Enterobacterales",     FALSE,
  "K. aerogenes",        "Klebsiella",           "Enterobacterales",     FALSE,
  "E. coli",              "Escherichia-Shigella", "Enterobacterales",     FALSE,
  "S. aureus",           "Staphylococcus",       "non-Enterobacterales", FALSE,
  "CoNS",                "Staphylococcus",       "non-Enterobacterales", TRUE,
  "Acinetobacter spp",   "Acinetobacter",        "non-Enterobacterales", FALSE,
  "Streptococcus spp",   "Streptococcus",        "non-Enterobacterales", FALSE,
  "P. aeruginosa",        "Pseudomonas",          "non-Enterobacterales", FALSE
)

sepsis_ids  <- sample_names(phy_16S_rel_gen_filtered)[
  sample_data(phy_16S_rel_gen_filtered)$group == "sepsis"]
control_ids <- sample_names(phy_16S_rel_gen_filtered)[
sample_data(phy_16S_rel_gen_filtered)$group == "control"]

# isolates without genus-level identification, dropped on purpose
bc_excluded <- c("Unidentified GNR")

# every isolate in the metadata must be either mapped or explicitly excluded
bc_seen <- meta %>%
  filter(group == "Sepsis") %>%
  pull(blood_culture_isolate) %>%
  as.character() %>% str_trim() %>% na.omit() %>% unique()
stopifnot("unmapped blood culture isolate" =
            all(bc_seen %in% c(bc_map$organism, bc_excluded)))

In [ ]:
bc_raw <- meta %>%
  filter(group == "Sepsis") %>%
  transmute(sample_id = sid_norm(study_id),
            organism  = str_trim(as.character(blood_culture_isolate))) %>%
  filter(sample_id %in% sepsis_ids)

bc_cases <- bc_raw %>%
  filter(!is.na(organism), !organism %in% bc_excluded) %>%
  left_join(bc_map, by = "organism") %>%
  arrange(stratum, silva_genus, sample_id)

stopifnot(!any(is.na(bc_cases$silva_genus)))

In [ ]:
# sample x genus matrix of relative abundances, zero-filled
abund <- phy_16S_rel_gen_filtered %>%
  psmelt() %>%
  as_tibble() %>%
  select(Sample, genus, Abundance) %>%
  group_by(Sample, genus) %>%
  summarise(Abundance = sum(Abundance), .groups = "drop") %>%
  pivot_wider(names_from = genus, values_from = Abundance, values_fill = 0) %>%
  column_to_rownames("Sample") %>%
  as.matrix()

# a target genus absent from the whole dataset is informative, not an error:
# it enters as an all-zero column and contributes a rank of exactly 0.5
missing_genera <- setdiff(unique(bc_cases$silva_genus), colnames(abund))
if (length(missing_genera)) {
  warning("target genera absent from the dataset (check SILVA spelling): ",
          toString(missing_genera))
  pad <- matrix(0, nrow(abund), length(missing_genera),
                dimnames = list(rownames(abund), missing_genera))
  abund <- cbind(abund, pad)
}

stopifnot(all(bc_cases$sample_id %in% rownames(abund)),
          all(sepsis_ids %in% rownames(abund)),
          all(control_ids %in% rownames(abund)))

### Permutation test


In [ ]:
# Gut-blood concordance: is the blood culture genus more abundant in the child's
# own rectal swab than in children who had a different pathogen (test A) or in
# controls (test B)?
#
# Statistic, per stratum s:
#   T_s = (1 / m_s) * sum over cases i in stratum s of R_i
#   R_i = ( #{j in Ref(i): x_j,g(i) < x_i,g(i)} + 0.5 * #{j in Ref(i): x_j,g(i) = x_i,g(i)} )
#         / |Ref(i)|
#   Ref(i) = sepsis children whose blood culture genus differs from g(i),
#            minus children with an unidentified gram-negative rod when g(i) is
#            itself a gram-negative rod (their isolate could be g(i))
# R_i is the percentile of the case within the reference distribution of its own
# pathogen genus; ties count half, so the null expectation is 0.5 also when most
# values are zero. Unit of observation is the child, so genera with more cases
# contribute more to T_s.

In [ ]:
# stratum and gram-negative status are functions of the genus, not of the species,
# because 16S does not resolve below genus
genus_info <- bc_map %>%
  distinct(silva_genus, stratum) %>%
  mutate(gram_neg_rod = silva_genus %in% c("Klebsiella", "Escherichia-Shigella",
                                           "Acinetobacter", "Pseudomonas"))
stopifnot(!any(duplicated(genus_info$silva_genus)))


# sequenced sepsis children whose isolate is an unidentified gram-negative rod;
# they are not cases but must leave the reference of gram-negative focal genera
gnr_ambiguous_ids <- meta %>%
  filter(group == "Sepsis",
         str_trim(as.character(blood_culture_isolate)) == "Unidentified GNR") %>%
  pull(study_id) %>% sid_norm() %>% intersect(sepsis_ids)

In [ ]:
genus_info

In [ ]:
# R_i: normalised mid-rank of one focal value within its reference vector
mid_rank <- function(x_focal, x_ref) {
  (sum(x_ref < x_focal) + 0.5 * sum(x_ref == x_focal)) / length(x_ref)
}

# Ref(i) for test A: drop every case carrying the focal genus (including i
# itself) and, for gram-negative focal genera, the ambiguous GNR children
ref_ids_A <- function(focal_genus, case_ids, case_genus, pool_ids) {
  drop <- case_ids[case_genus == focal_genus]
  if (genus_info$gram_neg_rod[match(focal_genus, genus_info$silva_genus)])
    drop <- c(drop, gnr_ambiguous_ids)
  setdiff(pool_ids, drop)
}

# T_s: mean of R_i over the cases of one stratum. exclude_same_genus = FALSE is
# test B, where the reference is the control group and no exclusion applies.
stat_T <- function(case_ids, case_genus, target_stratum, pool_ids, abund,
                   metric = "rank", exclude_same_genus = TRUE) {
  case_stratum <- genus_info$stratum[match(case_genus, genus_info$silva_genus)]
  keep <- which(case_stratum == target_stratum)
  if (!length(keep)) return(NA_real_)
  vals <- vapply(keep, function(k) {
    g   <- case_genus[k]
    ref <- if (exclude_same_genus) ref_ids_A(g, case_ids, case_genus, pool_ids)
           else setdiff(pool_ids, case_ids[k])
    if (!length(ref)) return(NA_real_)
    xf <- abund[case_ids[k], g]
    xr <- abund[ref, g]
    if (metric == "rank") mid_rank(xf, xr) else as.numeric(xf > 0) - mean(xr > 0)
  }, numeric(1))
  mean(vals, na.rm = TRUE)
}

# one-sided Monte Carlo p-value; the observed assignment counts as one of the
# possible ones, which keeps the test exact
perm_pvalue <- function(obs, null) (1 + sum(null >= obs)) / (length(null) + 1)


In [ ]:

# test A: the genus labels are reshuffled across cases, so the multiset of
# genera and therefore both stratum sizes are preserved. Ref(i) is recomputed
# from the assigned label in every permutation, which is what makes the null
# distribution the one of the procedure actually applied.
perm_test_A <- function(case_tbl, abund, pool_ids, target_stratum,
                        metric = "rank", B = 100000, seed = 123) {
  set.seed(seed)
  ids <- case_tbl$sample_id
  obs <- stat_T(ids, case_tbl$silva_genus, target_stratum, pool_ids, abund, metric)
  null <- vapply(seq_len(B), function(b) {
    stat_T(ids, sample(case_tbl$silva_genus), target_stratum, pool_ids, abund, metric)
  }, numeric(1))
  null <- null[!is.na(null)]
  list(test = "A", stratum = target_stratum, metric = metric,
       m = sum(genus_info$stratum[match(case_tbl$silva_genus, genus_info$silva_genus)] ==
                 target_stratum),
       statistic = obs, p_value = perm_pvalue(obs, null),
       null_mean = mean(null), null_sd = sd(null), null = null)
}

In [ ]:
# test B: reference is the control group. Group membership is randomised over
# all children first, then the observed genus labels are handed to a random
# subset of the new sepsis group, so group and pathogen identity are jointly
# independent of composition. 
perm_test_B <- function(case_tbl, abund, sepsis_ids, control_ids, target_stratum,
                        metric = "rank", B = 100000, seed = 123) {
  set.seed(seed)
  obs <- stat_T(case_tbl$sample_id, case_tbl$silva_genus, target_stratum,
                control_ids, abund, metric, exclude_same_genus = FALSE)
  pool   <- c(sepsis_ids, control_ids)
  n_sep  <- length(sepsis_ids)
  n_case <- nrow(case_tbl)
  null <- vapply(seq_len(B), function(b) {
    perm     <- sample(pool)
    new_sep  <- perm[seq_len(n_sep)]
    new_ctrl <- perm[(n_sep + 1L):length(perm)]
    stat_T(sample(new_sep, n_case), case_tbl$silva_genus, target_stratum,
           new_ctrl, abund, metric, exclude_same_genus = FALSE)
  }, numeric(1))
  null <- null[!is.na(null)]
  list(test = "B", stratum = target_stratum, metric = metric,
       m = sum(genus_info$stratum[match(case_tbl$silva_genus, genus_info$silva_genus)] ==
                 target_stratum),
       statistic = obs, p_value = perm_pvalue(obs, null),
       null_mean = mean(null), null_sd = sd(null), null = null)
}

summarise_perm <- function(...) {
  purrr::map(list(...), ~ tibble(test = .x$test, stratum = .x$stratum,
                                 metric = .x$metric, m = .x$m,
                                 statistic = .x$statistic, p_value = .x$p_value,
                                 null_mean = .x$null_mean, null_sd = .x$null_sd)) %>%
    purrr::list_rbind()
}

In [ ]:

stopifnot(all(bc_cases$silva_genus %in% colnames(abund)),
          all(bc_cases$sample_id %in% rownames(abund)),
          all(sepsis_ids %in% rownames(abund)),
          all(control_ids %in% rownames(abund)))

In [ ]:
# confirmatory test: A, Enterobacterales stratum, rank metric
res_A_gut  <- perm_test_A(bc_cases, abund, sepsis_ids, "Enterobacterales")
# secondary tests
res_A_gut_d <- perm_test_A(bc_cases, abund, sepsis_ids, "Enterobacterales", metric = "detect")
res_A_not  <- perm_test_A(bc_cases, abund, sepsis_ids, "non-Enterobacterales")
res_B_gut  <- perm_test_B(bc_cases, abund, sepsis_ids, control_ids, "Enterobacterales")
res_B_not  <- perm_test_B(bc_cases, abund, sepsis_ids, control_ids, "non-Enterobacterales")

perm_summary <- summarise_perm(res_A_gut, res_A_gut_d, res_A_not, res_B_gut, res_B_not)
write.csv(perm_summary, file.path(out_dir, "concordance_permutation_tests.csv"),
          row.names = FALSE)
perm_summary

In [ ]:
# per-case detail: focal abundance, its mid-rank, and rank within the own sample
case_detail <- bc_cases %>%
  mutate(rel_abundance = abund[cbind(match(sample_id, rownames(abund)),
                                     match(silva_genus, colnames(abund)))],
         rank_vs_sepsis = vapply(seq_len(n()), function(k) {
           ref <- ref_ids_A(silva_genus[k], sample_id, silva_genus, sepsis_ids)
           mid_rank(abund[sample_id[k], silva_genus[k]], abund[ref, silva_genus[k]])
         }, numeric(1)),
         rank_vs_control = vapply(seq_len(n()), function(k) {
           mid_rank(abund[sample_id[k], silva_genus[k]], abund[control_ids, silva_genus[k]])
         }, numeric(1)),
         within_rank = vapply(seq_len(n()), function(k)
           rank(-abund[sample_id[k], ], ties.method = "min")[silva_genus[k]], numeric(1))) %>%
  arrange(stratum, organism, desc(rel_abundance))

write.csv(case_detail, file.path(out_dir, "concordance_case_detail.csv"), row.names = FALSE)
case_detail

In [ ]:
# display ids for the supplement: reuse the crosswalk written by the composition
# figure, so both use the same numbering
crosswalk <- read.csv(file.path(out_dir, "sample_id_crosswalk.csv"),
                      stringsAsFactors = FALSE) %>%
  select(sample_id, display)

case_detail_pub <- case_detail %>%
  left_join(crosswalk, by = "sample_id") %>%
  select(infant = display, organism, genus = silva_genus, stratum,
         rel_abundance, rank_vs_sepsis, rank_vs_control, within_rank) %>%
  arrange(stratum, organism, desc(rel_abundance))

stopifnot(!any(is.na(case_detail_pub$infant)))

write.csv(case_detail_pub, file.path(out_dir, "concordance_case_detail_pub.csv"),
          row.names = FALSE)
case_detail_pub

In [ ]:
# per-case concordance: mid-rank of the own pathogen genus against sepsis infants
# with a different pathogen. 0.5 is the null expectation, point size shows the
# absolute relative abundance behind each rank.
fig3_df <- case_detail %>%
  left_join(crosswalk, by = "sample_id") %>%
  mutate(stratum = factor(stratum, levels = c("Enterobacterales", "non-Enterobacterales"),
                          labels = c("Enterobacterales",
                                     "Non-Enterobacterales")),
         label = paste0(display, " \u00b7 ", organism),
         label = factor(label, levels = rev(unique(
           label[order(stratum, organism, -rel_abundance)]))))

fig3 <- ggplot(fig3_df, aes(x = rank_vs_sepsis, y = label)) +
  geom_vline(xintercept = 0.5, linetype = "dashed", linewidth = 0.3, colour = "grey40") +
  geom_point(aes(size = rel_abundance), colour = "#45458b", alpha = 0.85) +
  facet_grid(stratum ~ ., scales = "free_y", space = "free_y") +
  scale_x_continuous(limits = c(0, 1), breaks = seq(0, 1, 0.25)) +
  scale_size_area(max_size = 4, labels = scales::percent_format(accuracy = 1)) +
  labs(x = "Normalized mid-rank of own blood culture genus",
       y = NULL, size = "Relative abundance") +
  theme_minimal(base_size = 8) +
    guides(size = guide_legend(title.position = "left", nrow = 1,
                             label.position = "bottom")) +
  theme(panel.grid.minor = element_blank(),
        panel.grid.major.y = element_line(linewidth = 0.2),
        strip.text.y = element_text(face = "bold", angle = 0),
        legend.position = "bottom",
        legend.key.size = unit(3, "mm"))

ggsave(file.path(out_dir, "figure3_concordance.pdf"), fig3,
       width = 120, height = 110, units = "mm", device = cairo_pdf)
ggsave(file.path(out_dir, "figure3_concordance.tiff"), fig3,
       width = 120, height = 110, units = "mm", dpi = 600,
       compression = "lzw", bg = "white")
fig3

### Sensitivity analysis

In [ ]:

# sensitivity: drop CoNS cases, the most frequent blood culture contaminant
bc_no_cons <- bc_cases %>% filter(!is_cons)
res_A_gut_nocons <- perm_test_A(bc_no_cons, abund, sepsis_ids, "Enterobacterales")
res_A_not_nocons <- perm_test_A(bc_no_cons, abund, sepsis_ids, "non-Enterobacterales")

summarise_perm(res_A_gut_nocons, res_A_not_nocons)



In [ ]:
# sensitivity: leave one case out of the confirmatory stratum, to show how much
# the result depends on single children
loo_gut <- purrr::map_dfr(which(bc_cases$stratum == "Enterobacterales"), function(k) {
  r <- perm_test_A(bc_cases[-k, ], abund, sepsis_ids, "Enterobacterales", B = 10000)
  tibble(dropped = bc_cases$sample_id[k], organism = bc_cases$organism[k],
         m = r$m, statistic = r$statistic, p_value = r$p_value)
})
loo_gut

In [ ]:
write.csv(summarise_perm(res_A_gut_nocons, res_A_not_nocons),
          file.path(out_dir, "concordance_sensitivity_nocons.csv"), row.names = FALSE)
write.csv(loo_gut, file.path(out_dir, "concordance_leave_one_out.csv"), row.names = FALSE)

### Null distribution for confirmatory test


In [ ]:
# null distribution of the confirmatory test, for the supplement
p_null <- ggplot(tibble(T_null = res_A_gut$null), aes(T_null)) +
  geom_histogram(bins = 60, fill = "grey70", colour = NA) +
  geom_vline(xintercept = res_A_gut$statistic, linewidth = 0.6) +
  geom_vline(xintercept = 0.5, linetype = "dashed", linewidth = 0.3) +
  labs(x = "Permuted test statistic (Enterobacterales stratum)", y = "Count") +
  theme_minimal(base_size = 8)

ggsave(file.path(out_dir, "nulldistribution.pdf"), p_null,
       width = 85, height = 60, units = "mm", device = cairo_pdf)
ggsave(file.path(out_dir, "nulldistribution.tiff"), p_null,
       width = 85, height = 60, units = "mm", dpi = 600,
       compression = "lzw", bg = "white")  
p_null

## Comparison between strata

In [ ]:
# does concordance differ between strata?
# R_i is held fixed per case and only the stratum labels are permuted, so the null
# hypothesis is that R_i is exchangeable between strata. With few cases all
# assignments are enumerated, which makes the p-value exact.
# The bootstrap interval resamples cases within strata, conditional on R_i.
stratum_diff_test <- function(case_tbl, r_col, primary_stratum, B_boot = 10000, seed = 123) {
  r   <- case_tbl[[r_col]]
  in1 <- case_tbl$stratum == primary_stratum
  n1  <- sum(in1)
  d_obs  <- mean(r[in1]) - mean(r[!in1])
  idx    <- combn(length(r), n1)
  d_null <- apply(idx, 2, function(k) mean(r[k]) - mean(r[-k]))
  eps    <- 1e-12
  set.seed(seed)
  d_boot <- replicate(B_boot, mean(sample(r[in1], replace = TRUE)) -
                              mean(sample(r[!in1], replace = TRUE)))
  ci <- unname(quantile(d_boot, c(0.025, 0.975)))
  tibble(reference     = r_col,
         m_primary     = n1,
         m_other       = sum(!in1),
         T_primary     = mean(r[in1]),
         T_other       = mean(r[!in1]),
         D             = d_obs,
         ci_lower      = ci[1],
         ci_upper      = ci[2],
         p_one_sided   = mean(d_null >= d_obs - eps),
         p_two_sided   = mean(abs(d_null) >= abs(d_obs) - eps),
         n_assignments = ncol(idx))
}

primary_stratum <- "Enterobacterales"

stratum_diff <- bind_rows(
  stratum_diff_test(case_detail, "rank_vs_sepsis", primary_stratum),
  stratum_diff_test(case_detail, "rank_vs_control", primary_stratum),
  stratum_diff_test(filter(case_detail, !is_cons), "rank_vs_sepsis", primary_stratum)
) %>%
  mutate(analysis = c("test A", "test B", "test A without CoNS"), .before = 1)

# stratum means must reproduce the statistics of the main permutation tests
stopifnot(isTRUE(all.equal(stratum_diff$T_primary[1], res_A_gut$statistic)),
          isTRUE(all.equal(stratum_diff$T_other[1],   res_A_not$statistic)))

write.csv(stratum_diff, file.path(out_dir, "concordance_stratum_difference.csv"), row.names = FALSE)
stratum_diff

# Stool cultures: culture-based colonisation with resistant gram negatives

In [ ]:
#ESBL screenings

stool_long <- meta %>%
  filter(group == "Sepsis") %>%
  transmute(sample_id = sid_norm(study_id),
            across(c(stool_c1, stool_c2, stool_c3, stool_c4),
                   ~ na_if(str_squish(as.character(.x)), "Not done"))) %>%
  filter(sample_id %in% sepsis_ids) %>%
  pivot_longer(starts_with("stool_c"), names_to = "visit", values_to = "raw") %>%
  mutate(visit = as.integer(str_remove(visit, "stool_c"))) %>%
  filter(!is.na(raw)) %>%
  separate_longer_delim(raw, delim = " and ") %>%
  mutate(raw = str_squish(raw),
         key = norm_key(raw),
         stool_genus = if_else(key %in% c("nbg", "nogrowth"), NA_character_,
                               bc_map$silva_genus[match(key, norm_key(bc_map$organism))]))

stopifnot(!any(is.na(stool_long$stool_genus) & !stool_long$key %in% c("nbg", "nogrowth")))

# one row per infant and stool genus, with the matching 16S abundance
stool_16s <- stool_long %>%
  filter(!is.na(stool_genus), stool_genus %in% colnames(abund)) %>%
  distinct(sample_id, stool_genus) %>%
  mutate(stool_genus_16S = abund[cbind(sample_id, stool_genus)])

# supplement table: blood culture, susceptibility, stool culture, 16S
stool_table <- bc_cases %>%
  select(sample_id, blood_organism = organism, blood_genus = silva_genus, stratum) %>%
  mutate(blood_genus_16S = abund[cbind(sample_id, blood_genus)]) %>%
  left_join(meta %>%
              filter(group == "Sepsis") %>%
              transmute(sample_id = sid_norm(study_id),
                        ceftriaxone = as.character(ceftriaxone_sir),
                        ceftazidime = as.character(ceftazidime_sir),
                        esbl = na_if(str_squish(as.character(esbl)), "")),
            by = "sample_id") %>%
  left_join(stool_long %>%
              filter(!is.na(stool_genus)) %>%
              group_by(sample_id) %>%
              summarise(stool_genera = paste(sort(unique(stool_genus)), collapse = ", "),
                        stool_done = TRUE, .groups = "drop"),
            by = "sample_id") %>%
  mutate(stool_done = sample_id %in% stool_long$sample_id,
         blood_genus_in_stool = if_else(stool_done,
                                        str_detect(replace_na(stool_genera, ""),
                                                   fixed(blood_genus)), NA)) %>%
  arrange(stratum, desc(blood_genus_16S))

write.csv(stool_table, file.path(out_dir, "stool_culture_table.csv"), row.names = FALSE)
stool_table


In [ ]:
# display ids for the supplement, same numbering as the composition figure
stool_table_pub <- stool_table %>%
  left_join(read.csv(file.path(out_dir, "sample_id_crosswalk.csv"),
                     stringsAsFactors = FALSE) %>%
              select(sample_id, display),
            by = "sample_id") %>%
  select(infant = display, blood_organism, blood_genus, stratum,
         ceftriaxone, ceftazidime, esbl,
         stool_done, stool_genera, blood_genus_in_stool, blood_genus_16S) %>%
  arrange(stratum, desc(blood_genus_16S))

stopifnot(!any(is.na(stool_table_pub$infant)))

write.csv(stool_table_pub, file.path(out_dir, "stool_culture_table_pub.csv"),
          row.names = FALSE)
stool_table_pub

In [ ]:
# MacConkey-based screening cannot grow gram-positive bacteria, so only
# gram-negative blood culture genera can be matched in stool
stool_table %>%
  filter(stool_done,
         blood_genus %in% genus_info$silva_genus[genus_info$gram_neg_rod]) %>%
  count(stratum, blood_genus_in_stool)  

comparison stool culture and 16S

In [ ]:
# stratum is a property of the genus, so it applies to stool isolates as well
stool_16s %>%
  left_join(genus_info %>% select(stool_genus = silva_genus, stratum),
            by = "stool_genus") %>%
  group_by(stratum, stool_genus) %>%
  summarise(n = n(),
            median = median(stool_genus_16S),
            min = min(stool_genus_16S),
            max = max(stool_genus_16S),
            n_zero = sum(stool_genus_16S == 0),
            .groups = "drop") %>%
  arrange(stratum, desc(median))

# SAVE

In [ ]:
# leave-one-out table with publication ids
loo_pub <- loo_gut %>%
  left_join(crosswalk %>% dplyr::select(sample_id, display), by = c("dropped" = "sample_id")) %>%
  dplyr::select(dropped = display, organism, m, statistic, p_value)
stopifnot(!anyNA(loo_pub$dropped))

# radEmu results for the supplement: one row per genus, sorted by p-value
tidy_emu_pub <- function(res) {
  res %>%
    transmute(genus          = label,
              family, phylum,
              n_samples      = non_zero_sample_count,
              low_prevalence = if_else(non_zero_sample_count < 5, "yes", "no"),
              log_fold       = round(estimate, 3),
              se             = round(se, 3),
              fold           = signif(fold, 3),
              fold_lower     = signif(fold_lower, 3),
              fold_upper     = signif(fold_upper, 3),
              p_value        = round(pval, 3),
              q_value        = round(qval, 3)) %>%
    arrange(p_value, genus)
}

yes_no <- function(x) case_when(is.na(x) ~ NA_character_, x ~ "yes", TRUE ~ "no")

emu_cols <- c(
  genus          = "Genus (SILVA 138.1); lineages not classified at genus level carry the lowest classified rank with the suffix _spc",
  family         = "Family",
  phylum         = "Phylum",
  n_samples      = "Number of samples (of 34) in which the genus was detected",
  low_prevalence = "yes if detected in fewer than five samples",
  log_fold       = "Estimated log fold-difference (natural logarithm), neonates with BSI versus healthy neonates",
  se             = "Standard error of log_fold",
  fold           = "Fold-difference, exp(log_fold)",
  fold_lower     = "Lower bound of the 95% confidence interval of fold",
  fold_upper     = "Upper bound of the 95% confidence interval of fold",
  p_value        = "p-value of the robust score test",
  q_value        = "Benjamini-Hochberg adjusted p-value"
)

r_note <- paste(
  "R is the normalized mid-rank of the blood culture genus in the neonate's own swab within the reference set:",
  "the proportion of reference neonates with a lower relative abundance, counting ties as one half (0.5 = chance).",
  "T is the mean of R across the cases of a stratum."
)

supp <- list(
  "Table S1" = list(
    data  = tidy_emu_pub(emu_16S_pc_res),
    title = "Differential abundance of genera between neonates with BSI and healthy neonates",
    note  = paste("radEmu model ~ group with healthy neonates as reference.",
                  "Fold-differences are expressed relative to the typical fold-difference across all genera."),
    cols  = emu_cols),

  "Table S2" = list(
    data  = tidy_emu_pub(emu_16S_pc_abx_res),
    title = "Differential abundance of genera adjusted for antibiotic exposure during pregnancy",
    note  = paste("radEmu model ~ group + antibiotic exposure during pregnancy with healthy neonates as reference.",
                  "Only the group effect is shown. Fold-differences are expressed relative to the typical",
                  "fold-difference across all genera."),
    cols  = emu_cols),

  "Table S3" = list(
    data  = case_detail_pub %>%
      transmute(neonate           = infant,
                blood_organism    = organism,
                blood_genus       = genus,
                stratum,
                rel_abundance_pct = round(100 * rel_abundance, 2),
                R_test_A          = round(rank_vs_sepsis, 3),
                R_test_B          = round(rank_vs_control, 3),
                rank_in_own_swab  = within_rank),
    title = "Relative abundance and ranks of the blood culture genus in the rectal swab of each neonate with BSI",
    note  = paste(r_note,
                  "Test A: reference = other neonates with BSI whose blood culture genus differed;",
                  "test B: reference = healthy neonates."),
    cols  = c(
      neonate           = "Study identifier of the neonate with BSI",
      blood_organism    = "Organism isolated from blood culture; CoNS, coagulase-negative staphylococci",
      blood_genus       = "SILVA genus corresponding to the blood culture isolate",
      stratum           = "Pathogen stratum (Enterobacterales or non-Enterobacterales)",
      rel_abundance_pct = "Relative abundance of the blood culture genus in the neonate's own rectal swab (%)",
      R_test_A          = "Normalized mid-rank R in test A",
      R_test_B          = "Normalized mid-rank R in test B",
      rank_in_own_swab  = "Rank of the blood culture genus among all genera in the neonate's own swab (1 = most abundant; tied genera share the lowest rank, so absent genera share the rank after the last detected genus)")),

  "Table S4" = list(
    data  = stool_table_pub %>%
      transmute(neonate                  = infant,
                blood_organism, blood_genus, stratum,
                ceftriaxone, ceftazidime,
                beta_lactamase           = case_when(
                  str_detect(str_to_lower(esbl), "ampc") ~ "AmpC",
                  str_to_lower(esbl) == "yes"            ~ "ESBL",
                  str_to_lower(esbl) == "no"             ~ "none",
                  TRUE                                   ~ NA_character_),
                screening_done           = yes_no(stool_done),
                screening_genera         = case_when(
                  !stool_done                                ~ NA_character_,
                  is.na(stool_genera) | stool_genera == ""   ~ "no growth",
                  TRUE                                       ~ stool_genera),
                blood_genus_in_screening = yes_no(blood_genus_in_stool),
                blood_genus_16S_pct      = round(100 * blood_genus_16S, 2)),
    title = "Blood culture isolate, antimicrobial susceptibility and culture-based rectal screening of each neonate with BSI",
    note  = paste("Rectal screening was performed on cefotaxime-supplemented (2 mg/L) MacConkey agar.",
                  "Susceptibility: S, susceptible; I, intermediate; R, resistant. Empty cells: not tested or not reported.",
                  "CoNS, coagulase-negative staphylococci."),
    cols  = c(
      neonate                  = "Study identifier of the neonate with BSI",
      blood_organism           = "Organism isolated from blood culture",
      blood_genus              = "SILVA genus corresponding to the blood culture isolate",
      stratum                  = "Pathogen stratum (Enterobacterales or non-Enterobacterales)",
      ceftriaxone              = "Susceptibility of the blood culture isolate to ceftriaxone",
      ceftazidime              = "Susceptibility of the blood culture isolate to ceftazidime",
      beta_lactamase           = "Beta-lactamase phenotype of the blood culture isolate; ESBL, extended-spectrum beta-lactamase; AmpC, AmpC beta-lactamase",
      screening_done           = "Culture-based rectal screening performed",
      screening_genera         = "Genera isolated by culture-based rectal screening",
      blood_genus_in_screening = "Blood culture genus isolated by culture-based rectal screening",
      blood_genus_16S_pct      = "Relative abundance of the blood culture genus in the 16S rRNA gene data of the rectal swab (%)")),

  "Table S5" = list(
    data  = loo_pub %>%
      transmute(excluded_neonate = dropped,
                blood_organism   = organism,
                m,
                T       = round(statistic, 3),
                p_value = round(p_value, 3)),
    title = "Leave-one-out analysis of the primary concordance test (test A, Enterobacterales stratum)",
    note  = paste(r_note, "Each case was excluded in turn; p-values from 10,000 permutations."),
    cols  = c(
      excluded_neonate = "Study identifier of the excluded neonate",
      blood_organism   = "Blood culture organism of the excluded neonate",
      m                = "Number of cases remaining in the stratum",
      T                = "Test statistic T after exclusion",
      p_value          = "One-sided permutation p-value")),

  "Table S6" = list(
    data  = bind_rows(
      perm_summary %>% mutate(analysis = "all cases"),
      summarise_perm(res_A_gut_nocons, res_A_not_nocons) %>% mutate(analysis = "CoNS cases removed")
    ) %>%
      transmute(analysis, test, stratum,
                T         = round(statistic, 3),
                statistic = if_else(metric == "detect", "detection", "mid-rank"),
                m,
                p_value   = round(p_value, 3),
                null_mean = round(null_mean, 3),
                null_sd   = round(null_sd, 3)) %>%
      relocate(statistic, m, .after = stratum),
    title = "Concordance between blood culture isolates and rectal microbiota: permutation tests",
    note  = paste(r_note,
                  "Test A: reference = other neonates with BSI whose blood culture genus differed;",
                  "test B: reference = healthy neonates. The primary analysis was test A with the mid-rank statistic",
                  "in the Enterobacterales stratum. For the detection-based statistic, R was replaced by the difference",
                  "between detection of the blood culture genus in the own swab (0 or 1) and its detection prevalence",
                  "in the reference set (null expectation 0). One-sided p-values from 100,000 permutations.",
                  "CoNS, coagulase-negative staphylococci."),
    cols  = c(
      analysis  = "Set of cases (all cases, or after removal of cases with CoNS)",
      test      = "Concordance test (A or B)",
      stratum   = "Pathogen stratum",
      statistic = "Type of test statistic (mid-rank or detection)",
      m         = "Number of cases in the stratum",
      T         = "Observed test statistic",
      p_value   = "One-sided permutation p-value",
      null_mean = "Mean of the test statistic under the permutation null distribution",
      null_sd   = "Standard deviation of the test statistic under the permutation null distribution")),

  "Table S7" = list(
    data  = stratum_diff %>%
      transmute(analysis,
                m_Ent         = m_primary,
                m_non         = m_other,
                T_Ent         = round(T_primary, 3),
                T_non         = round(T_other, 3),
                D             = round(D, 3),
                ci_lower      = round(ci_lower, 3),
                ci_upper      = round(ci_upper, 3),
                p_one_sided   = round(p_one_sided, 3),
                n_assignments),
    title = "Difference in concordance between Enterobacterales and non-Enterobacterales cases",
    note  = paste(r_note,
                  "D = T_Ent - T_non. Null hypothesis D <= 0 against D > 0; exact p-value from enumeration of all",
                  "assignments of stratum labels with R held fixed; 95% CI from a percentile bootstrap within strata",
                  "(10,000 resamples)."),
    cols  = c(
      analysis      = "Concordance test compared between strata (test A, test B, or test A without CoNS cases)",
      m_Ent         = "Number of Enterobacterales cases",
      m_non         = "Number of non-Enterobacterales cases",
      T_Ent         = "Test statistic T in the Enterobacterales stratum",
      T_non         = "Test statistic T in the non-Enterobacterales stratum",
      D             = "Difference T_Ent - T_non",
      ci_lower      = "Lower bound of the 95% bootstrap confidence interval of D",
      ci_upper      = "Upper bound of the 95% bootstrap confidence interval of D",
      p_one_sided   = "Exact one-sided p-value",
      n_assignments = "Number of possible assignments of stratum labels enumerated for the exact test"))
)
# order and number the tables as they are first cited in the results
supp <- supp[c("Table S1", "Table S2", "Table S3", "Table S6", "Table S5", "Table S7", "Table S4")]
names(supp) <- paste0("Supplementary Table ", seq_along(supp))

# every column of every table must be documented, and nothing else
purrr::iwalk(supp, function(x, nm) {
  undocumented <- setdiff(names(x$data), names(x$cols))
  unused       <- setdiff(names(x$cols), names(x$data))
  if (length(undocumented) || length(unused))
    stop(nm, ": undocumented columns: ", toString(undocumented),
         "; descriptions without column: ", toString(unused))
})

# legends sheet: title, note and one row per column for each table
legends <- purrr::imap_dfr(supp, function(x, nm) {
  bind_rows(
    tibble(Table = nm, Item = "Title", Description = x$title),
    tibble(Table = nm, Item = "Note",  Description = x$note),
    tibble(Table = nm, Item = names(x$cols), Description = unname(x$cols))
  )
})

write_xlsx(c(list(Legends = legends), purrr::map(supp, "data")),
           file.path(out_dir, "supplementary_tables.xlsx"))

legends %>% filter(Item == "Title")

In [ ]:
writeLines(capture.output(sessionInfo()), file.path(out_dir, "sessionInfo.txt"))